In [ ]:
!pip install -q langchain langchain-core langchain-community langchain-google-genai
!pip install -q langchain-groq langchain-huggingface langchain-text-splitters
!pip install -q faiss-cpu sentence-transformers gradio
print("✅ All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 41.8 MB/s eta 0:00:00
✅ All packages installed!


In [ ]:
# ============================================================
# COMPLETE SETUP - Run this after every Colab session restart
# ============================================================
import os
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── Step 1: Load API Keys ────────────────────────────────────
try:
    from google.colab import userdata
    GEMINI_API_KEY       = userdata.get('GEMINI_API_KEY')
    GROQ_API_KEY         = userdata.get('GROQ_API_KEY')
    SERP_API_KEY         = userdata.get('SERP_API_KEY')
    WEATHERSTACK_API_KEY = userdata.get('WEATHERSTACK_API_KEY')
    EXCHANGERATE_API_KEY = userdata.get('EXCHANGERATE_API_KEY')
    print("✅ Step 1: API Keys loaded!")
except:
    GEMINI_API_KEY = GROQ_API_KEY = SERP_API_KEY = ""
    WEATHERSTACK_API_KEY = EXCHANGERATE_API_KEY = ""
    print("⚠️  Add your API keys manually!")

os.environ["GEMINI_API_KEY"]  = GEMINI_API_KEY
os.environ["SERP_API_KEY"] = SERP_API_KEY

# ── Step 2: Initialize LLM (Groq) ───────────────────────────
from langchain_groq import ChatGroq

GROQ_MODELS = [
    "llama3-70b-8192",
    "llama-3.1-8b-instant",
    "mixtral-8x7b-32768",
    "gemma2-9b-it",
]

llm = None
for model_name in GROQ_MODELS:
    try:
        test_llm = ChatGroq(
            model=model_name,
            temperature=0.7,
            groq_api_key=GROQ_API_KEY
        )
        test_llm.invoke("Hi")
        llm = test_llm
        print(f"✅ Step 2: LLM ready → {model_name}")
        break
    except Exception as e:
        print(f"❌ {model_name} failed: {str(e)[:60]}")

# ── Step 3: Build RAG Knowledge Base ────────────────────────
travel_knowledge = [
    """
    VISA INFORMATION:
    Most countries require a valid passport with at least 6 months validity.
    Schengen visa covers 26 European countries with a single visa.
    USA requires ESTA for visa-waiver countries or a B1/B2 visa.
    India e-Visa is available for 170+ countries online.
    Always apply for visas 4-6 weeks in advance.
    """,
    """
    TRAVEL BUDGET TIPS:
    Southeast Asia (Thailand, Vietnam, Bali) is budget-friendly: $30-60/day including accommodation.
    Western Europe averages $100-200/day for mid-range travel.
    Japan costs $80-150/day; best visited in spring (cherry blossoms) or autumn (fall colors).
    Book flights 6-8 weeks in advance for best prices.
    Use incognito mode when searching flights to avoid price tracking.
    """,
    """
    PACKING ESSENTIALS:
    Always carry: passport, travel insurance docs, emergency cash.
    Universal power adapter for international travel.
    Download offline maps (Google Maps) before arriving.
    Carry copies of all important documents separately.
    Travel insurance is strongly recommended for all trips.
    """,
    """
    TOP DESTINATIONS 2024:
    Europe: Paris, Rome, Barcelona, Amsterdam, Prague
    Asia: Tokyo, Bali, Bangkok, Singapore, Kyoto
    Americas: New York, Machu Picchu, Patagonia, Costa Rica
    Middle East: Dubai, Petra, Istanbul, Muscat
    Best time to visit Bali: April-October (dry season)
    Best time to visit Europe: May-September
    """,
    """
    INDIA TRAVEL INFORMATION:
    India has diverse climates - plan based on region and season.
    North India (Delhi, Agra, Jaipur) Golden Triangle: October-March is ideal.
    Kerala backwaters: November-February is best.
    Goa beaches: November-February (peak season), avoid monsoon June-September.
    Himachal Pradesh and Ladakh: May-September.
    Budget: Rs.2000-5000/day for mid-range domestic travel.
    Book trains on IRCTC at least 2-3 months in advance.
    """,
    """
    FLIGHT BOOKING TIPS:
    Best days to book: Tuesday and Wednesday for cheaper fares.
    Use Google Flights, Skyscanner, or Kayak to compare prices.
    Flexible date search can save 20-40% on airfare.
    Consider nearby airports for cheaper options.
    Budget airlines: IndiGo, SpiceJet (India), Ryanair (Europe), AirAsia (Asia).
    Always check baggage allowance before booking.
    """,
    """
    HOTEL BOOKING TIPS:
    Book directly with hotels for better rates and flexibility.
    Compare on Booking.com, Airbnb, Hotels.com, MakeMyTrip.
    Hostels are great for solo travelers ($10-25/night).
    Mid-range hotels: $40-100/night globally.
    Check cancellation policies, especially for long trips.
    Location matters - staying central saves on transport costs.
    """
]

docs       = [Document(page_content=text) for text in travel_knowledge]
splitter   = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(docs)

print("⏳ Step 3: Loading embeddings (first time ~30 seconds)...")
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"✅ Step 3: RAG Knowledge base ready with {len(split_docs)} chunks!")

# ── Step 4: Build RAG Chain ──────────────────────────────────
TRAVEL_PROMPT = PromptTemplate(
    template="""
You are an expert AI Travel Concierge with deep knowledge about travel worldwide.
You are friendly, helpful, and provide practical, actionable travel advice.

Use the following travel knowledge to answer the question:
{context}

Question: {question}

Instructions:
- Be specific and helpful
- Include practical tips when relevant
- Mention costs/budgets when relevant
- Keep response concise but complete

Answer:
""",
    input_variables=["context", "question"]
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

travel_qa_chain = (
    {
        "context" : retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | TRAVEL_PROMPT
    | llm
    | StrOutputParser()
)

def basic_travel_chat(question: str) -> str:
    try:
        return travel_qa_chain.invoke(question)
    except Exception as e:
        return f"Error: {str(e)}"

print("✅ Step 4: RAG Chain built!")

# ── Step 5: Quick Test ───────────────────────────────────────
print("\n🧪 Quick test...")
print("🤖", basic_travel_chat("Best time to visit Bali?"))

print("\n" + "="*50)
print("🎉 ALL STEPS COMPLETE! Everything is ready.")
print("="*50)
print("✅ llm       → LLM model")
print("✅ retriever → RAG knowledge base")
print("✅ travel_qa_chain → Chatbot chain")
print("\n▶️  Now run the Gradio UI cell below!")

✅ Step 1: API Keys loaded!
❌ llama3-70b-8192 failed: Error code: 400 - {'error': {'message': 'The model `llama3-7
✅ Step 2: LLM ready → llama-3.1-8b-instant
⏳ Step 3: Loading embeddings (first time ~30 seconds)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Step 3: RAG Knowledge base ready with 7 chunks!
✅ Step 4: RAG Chain built!

🧪 Quick test...
🤖 The best time to visit Bali is during the dry season, which runs from April to October. This period offers comfortable temperatures, minimal rainfall, and clear skies, making it ideal for exploring the island's beaches, temples, and lush landscapes.

To make the most of your trip, consider the following practical tips:

- Book your accommodations and flights in advance, especially during peak season (June to September) when prices tend to be higher.
- Budget around $30-60/day for a mid-range stay, including accommodation and meals.
- Plan your activities and tours in advance to avoid crowds and make the most of your time on the island.
- Don't forget to try the local cuisine, including nasi goreng, mie goreng, and fresh seafood.

Keep in mind that the shoulder season (April to May and September to October) can be a great time to visit Bali, as prices tend to be lower and the crowds are small

**WEEK 1-2 TASK**

In [ ]:
!pip install -q langchain langchain-google-genai langchain-community
!pip install -q google-generativeai
!pip install -q chromadb  # Vector database for RAG
!pip install -q gradio    # UI that works inside Colab
!pip install -q requests  # For API calls
!pip install -q python-dotenv  # For managing API keys
!pip install -q google-search-results  # SERP API wrapper
!pip install -q langchain-chroma
!pip install -q sentence-transformers  # For embeddings
!pip install -q faiss-cpu  # Fast vector search
!pip install -q langchain-core
!pip install -U google-generativeai langchain-google-genai
!pip install -q langchain-text-splitters
print("✅ Installed!")
!pip install -q sentence-transformers langchain-huggingface
print("✅ Done!")
!pip install -q --upgrade langchain-google-genai google-generativeai langchain-groq
print("✅ Upgraded! Now click Runtime → Restart Session, then continue.")
!pip install -q langchain langchain-core langchain-community
print("✅ Done!")
!pip install -q langchain langchain-core langchain-community langchain-google-genai
print("✅ Done!")

print("✅ All packages installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [ ]:
# ============================================================
# API KEY CONFIGURATION
# ============================================================
# Option 1: Using Colab Secrets (RECOMMENDED - more secure)
# Go to the key icon (🔑) on the left sidebar → add these secrets:
#   GEMINI_API_KEY
#   SERP_API_KEY
#   GROQ_API_KEY
#   WEATHERSTACK_API_KEY
#   EXCHANGERATE_API_KEY

import os

try:
    from google.colab import userdata

    # Load keys from Colab Secrets
    GEMINI_API_KEY       = userdata.get('GEMINI_API_KEY')
    SERP_API_KEY         = userdata.get('SERP_API_KEY')
    GROQ_API_KEY         = userdata.get('GROQ_API_KEY')
    WEATHERSTACK_API_KEY = userdata.get('WEATHERSTACK_API_KEY')
    EXCHANGERATE_API_KEY = userdata.get('EXCHANGERATE_API_KEY')

    print("✅ Keys loaded from Colab Secrets")

except Exception:
    # Option 2: Paste keys directly (only for local testing)
    print("⚠️  Colab Secrets not found. Using manual entry.")
    GEMINI_API_KEY       = "GEMINI_API_KEY"
    SERP_API_KEY         = "SERP_API_KEY"
    GROQ_API_KEY         = "GROQ_API_KEY"
    WEATHERSTACK_API_KEY = "WEATHERSTACK_API_KEY"
    EXCHANGERATE_API_KEY = "EXCHANGERATE_API_KEY"

# Set as environment variables (required by LangChain)
os.environ["GEMINI_API_KEY"]        = GEMINI_API_KEY
os.environ["SERP_API_KEY"]       = SERP_API_KEY
os.environ["GROQ_API_KEY"]          = GROQ_API_KEY
os.environ["WEATHERSTACK_API_KEY"]  = WEATHERSTACK_API_KEY
os.environ["EXCHANGERATE_API_KEY"]  = EXCHANGERATE_API_KEY

# Quick validation - checks keys are not empty
keys = {
    "Gemini"        : GEMINI_API_KEY,
    "SERP"          : SERP_API_KEY,
    "Groq"          : GROQ_API_KEY,
    "Weatherstack"  : WEATHERSTACK_API_KEY,
    "ExchangeRate"  : EXCHANGERATE_API_KEY,
}
for name, key in keys.items():
    status = "✅" if key and key != f"YOUR_{name.upper()}_API_KEY_HERE" else "❌ Missing"
    print(f"{status} {name} API Key")

✅ Keys loaded from Colab Secrets
✅ Gemini API Key
✅ SERP API Key
✅ Groq API Key
✅ Weatherstack API Key
✅ ExchangeRate API Key


In [ ]:
# ============================================================
# LLM INITIALIZATION - FIXED (Groq primary, Gemini fallback)
# ============================================================
import os

try:
    from google.colab import userdata
    GEMINI_API_KEY       = userdata.get('GEMINI_API_KEY')
    GROQ_API_KEY         = userdata.get('GROQ_API_KEY')
    SERP_API_KEY         = userdata.get('SERP_API_KEY')
    WEATHERSTACK_API_KEY = userdata.get('WEATHERSTACK_API_KEY')
    EXCHANGERATE_API_KEY = userdata.get('EXCHANGERATE_API_KEY')
    print("✅ API Keys loaded!")
except:
    print("⚠️ Add your keys manually below")
    GEMINI_API_KEY       = "GEMINI_API_KEY"
    GROQ_API_KEY         = "GROQ_API_KEY"
    SERP_API_KEY         = "SERP_API_KEY"
    WEATHERSTACK_API_KEY = "WEATHERSTACK_API_KEY"
    EXCHANGERATE_API_KEY = "EXCHANGERATE_API_KEY"

os.environ["GEMINI_API_KEY"]  = GEMINI_API_KEY
os.environ["SERP_API_KEY"] = SERP_API_KEY

# ── Try Groq first (most reliable, fully free) ──────────────
from langchain_groq import ChatGroq

# Current working Groq models (as of 2025)
GROQ_MODELS = [
    "llama3-70b-8192",       # Best quality on Groq
    "llama-3.1-8b-instant",  # Fast and free
    "mixtral-8x7b-32768",    # Good alternative
    "gemma2-9b-it",          # Google's Gemma on Groq
]

llm = None

print("\n🔄 Trying Groq models...")
for model_name in GROQ_MODELS:
    try:
        print(f"⏳ Trying: {model_name} ...")
        test_llm = ChatGroq(
            model=model_name,
            temperature=0.7,
            groq_api_key=GROQ_API_KEY
        )
        response = test_llm.invoke("Say 'Travel Concierge ready!' in one sentence.")
        print(f"✅ '{model_name}' works! → {response.content}")
        llm = test_llm
        break
    except Exception as e:
        print(f"❌ '{model_name}' failed: {str(e)[:100]}")

# ── If Groq fails, try Gemini ───────────────────────────────
if llm is None:
    print("\n🔄 Groq failed. Trying Gemini models...")
    from langchain_google_genai import ChatGoogleGenerativeAI

    GEMINI_MODELS = [
        "gemini-2.0-flash",
        "gemini-1.5-flash",
        "gemini-1.0-pro",
    ]

    for model_name in GEMINI_MODELS:
        try:
            print(f"⏳ Trying: {model_name} ...")
            test_llm = ChatGoogleGenerativeAI(
                model=model_name,
                temperature=0.7,
                google_api_key=GEMINI_API_KEY
            )
            response = test_llm.invoke("Say 'Travel Concierge ready!' in one sentence.")
            print(f"✅ '{model_name}' works! → {response.content}")
            llm = test_llm
            break
        except Exception as e:
            print(f"❌ '{model_name}' failed: {str(e)[:100]}")

# ── Final status ────────────────────────────────────────────
if llm is None:
    print("\n❌ All models failed. Please check your API keys in Colab Secrets.")
else:
    print(f"\n🎉 LLM is ready! Using: {llm.model_name if hasattr(llm, 'model_name') else 'Selected model'}")
    print("✅ You can now proceed to the next cells!")

✅ API Keys loaded!

🔄 Trying Groq models...
⏳ Trying: llama3-70b-8192 ...
❌ 'llama3-70b-8192' failed: Error code: 400 - {'error': {'message': 'The model `llama3-70b-8192` has been decommissioned and is 
⏳ Trying: llama-3.1-8b-instant ...
✅ 'llama-3.1-8b-instant' works! → I'm your Travel Concierge, ready to assist you in planning an unforgettable journey.

🎉 LLM is ready! Using: llama-3.1-8b-instant
✅ You can now proceed to the next cells!


In [ ]:
# ============================================================
# BASIC RAG TRAVEL CHATBOT - FINAL FIXED VERSION
# ============================================================
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

TRAVEL_PROMPT_TEMPLATE = """
You are an expert AI Travel Concierge with deep knowledge about travel worldwide.
You are friendly, helpful, and provide practical, actionable travel advice.

Use the following travel knowledge to answer the question:
{context}

Question: {question}

Instructions:
- Be specific and helpful
- Include practical tips when relevant
- Mention costs/budgets when relevant
- If you don't know something specific, say so and give general advice
- Keep response concise but complete

Answer:
"""

TRAVEL_PROMPT = PromptTemplate(
    template=TRAVEL_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build RAG chain
travel_qa_chain = (
    {
        "context" : retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | TRAVEL_PROMPT
    | llm
    | StrOutputParser()
)

def basic_travel_chat(question: str) -> str:
    try:
        return travel_qa_chain.invoke(question)
    except Exception as e:
        return f"Error: {str(e)}"

# Test
print("🧪 Testing the basic travel chatbot...\n")
test_questions = [
    "What's the best time to visit Bali?",
    "Give me budget tips for Southeast Asia travel",
    "What documents do I need for international travel?"
]

for q in test_questions:
    print(f"❓ Q: {q}")
    print(f"🤖 A: {basic_travel_chat(q)}")
    print("-" * 60)

🧪 Testing the basic travel chatbot...

❓ Q: What's the best time to visit Bali?
🤖 A: The best time to visit Bali is from April to October, which is the dry season. This period offers pleasant weather with average temperatures ranging from 20°C to 30°C (68°F to 86°F). 

During this time, you can enjoy a range of outdoor activities like surfing, snorkeling, and exploring the island's beautiful beaches. Additionally, the dry season is ideal for visiting Bali's temples and attending cultural events like the Ubud Food Festival.

To make the most of your trip, consider the following practical tips:

- Book your accommodations and flights in advance to avoid peak season prices.
- Budget around $30-60 per day for mid-range travel, including accommodation.
- Plan your itinerary according to the island's festivals and events to experience the local culture.

Keep in mind that the dry season is peak tourist season, so expect larger crowds and higher prices for accommodations and flights. However,

In [ ]:
# ============================================================
# GRADIO CHAT INTERFACE
# ============================================================
import gradio as gr

# Chat history storage
chat_history = []

def chat_with_concierge(user_message, history):
    """Handle a chat message and return response."""
    if not user_message.strip():
        return "", history

    # Get response from our RAG chatbot
    bot_response = basic_travel_chat(user_message)

    # Add to history
    history.append((user_message, bot_response))
    return "", history

# Create the Gradio interface
with gr.Blocks(title="AI Travel Concierge", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""# ✈️ AI Travel Concierge
    ### Your intelligent travel planning assistant powered by Gemini AI
    Ask me anything about travel destinations, flights, hotels, budgets, and more!
    """)

    chatbot = gr.Chatbot(
        value=[(None, "👋 Hello! I'm your AI Travel Concierge. I can help you plan trips, suggest destinations, estimate budgets, and more. Where would you like to go?")],
        height=450,
        label="Travel Assistant"
    )

    with gr.Row():
        msg_input = gr.Textbox(
            placeholder="e.g. Plan a 7-day trip to Japan for 2 people...",
            label="Your question",
            scale=4
        )
        send_btn = gr.Button("Send ✈️", scale=1, variant="primary")

    # Sample question buttons
    gr.Markdown("**💡 Try these:**")
    with gr.Row():
        gr.Button("Best time to visit Bali?").click(
            lambda: "Best time to visit Bali?", outputs=msg_input
        )
        gr.Button("Budget for 7 days in Thailand?").click(
            lambda: "What's the budget for 7 days in Thailand?", outputs=msg_input
        )
        gr.Button("India trip packing list").click(
            lambda: "Give me a packing list for India trip", outputs=msg_input
        )

    # Connect buttons to chat function
    send_btn.click(chat_with_concierge, [msg_input, chatbot], [msg_input, chatbot])
    msg_input.submit(chat_with_concierge, [msg_input, chatbot], [msg_input, chatbot])

# Launch the interface
# share=True creates a public link you can share (valid for 72 hours)
demo.launch(share=True, debug=False)
print("\n🔗 Your Travel Concierge UI is live! Click the link above.")

/tmp/ipykernel_10939/1947053557.py:22: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AI Travel Concierge", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_10939/1947053557.py:29: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_10939/1947053557.py:29: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://79455960198f6ad426.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🔗 Your Travel Concierge UI is live! Click the link above.


**WEEK 3-4 TASK**

In [ ]:
# ============================================================
# MASTER SETUP - Run this first after every session restart
# ============================================================
import os
import requests
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from serpapi import GoogleSearch

# ── Step 1: API Keys ─────────────────────────────────────────
try:
    from google.colab import userdata
    GEMINI_API_KEY       = userdata.get('GEMINI_API_KEY')
    GROQ_API_KEY         = userdata.get('GROQ_API_KEY')
    SERP_API_KEY         = userdata.get('SERP_API_KEY')
    WEATHERSTACK_API_KEY = userdata.get('WEATHERSTACK_API_KEY')
    EXCHANGERATE_API_KEY = userdata.get('EXCHANGERATE_API_KEY')
    print("✅ Step 1: API Keys loaded!")
except:
    print("⚠️  Add keys manually!")

os.environ["SERPAPI_API_KEY"] = SERP_API_KEY

# ── Step 2: LLM ──────────────────────────────────────────────
GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "llama-3.1-70b-versatile",
    "llama-3.1-8b-instant",
    "mixtral-8x7b-32768",
    "gemma2-9b-it",
]
llm = None
for model in GROQ_MODELS:
    try:
        test = ChatGroq(model=model, temperature=0.7, groq_api_key=GROQ_API_KEY)
        test.invoke("Hi")
        llm = test
        print(f"✅ Step 2: LLM ready → {model}")
        break
    except Exception as e:
        print(f"❌ {model}: {str(e)[:50]}")

# ── Step 3: Tools ─────────────────────────────────────────────
CITY_WEATHER = {
    "london"    : {"temp":14, "feel":11, "desc":"Overcast/Light Rain",  "humidity":78, "wind":19},
    "paris"     : {"temp":16, "feel":13, "desc":"Partly Cloudy",        "humidity":72, "wind":15},
    "tokyo"     : {"temp":21, "feel":19, "desc":"Clear Sky",            "humidity":63, "wind":12},
    "bali"      : {"temp":29, "feel":33, "desc":"Sunny with Humidity",  "humidity":82, "wind":10},
    "bangkok"   : {"temp":33, "feel":38, "desc":"Hot & Humid",          "humidity":85, "wind":8},
    "singapore" : {"temp":30, "feel":35, "desc":"Partly Cloudy",        "humidity":84, "wind":11},
    "dubai"     : {"temp":36, "feel":40, "desc":"Sunny & Hot",          "humidity":48, "wind":14},
    "mumbai"    : {"temp":32, "feel":36, "desc":"Hazy & Humid",         "humidity":88, "wind":16},
    "delhi"     : {"temp":28, "feel":30, "desc":"Partly Cloudy",        "humidity":55, "wind":10},
    "new york"  : {"temp":18, "feel":15, "desc":"Mostly Clear",         "humidity":58, "wind":20},
    "sydney"    : {"temp":22, "feel":20, "desc":"Sunny",                "humidity":65, "wind":18},
    "rome"      : {"temp":20, "feel":18, "desc":"Sunny",                "humidity":60, "wind":12},
    "barcelona" : {"temp":22, "feel":20, "desc":"Clear & Sunny",        "humidity":62, "wind":14},
    "amsterdam" : {"temp":13, "feel":10, "desc":"Cloudy",               "humidity":80, "wind":22},
    "istanbul"  : {"temp":19, "feel":17, "desc":"Partly Cloudy",        "humidity":68, "wind":16},
    "seoul"     : {"temp":18, "feel":16, "desc":"Clear",                "humidity":55, "wind":13},
    "goa"       : {"temp":31, "feel":35, "desc":"Sunny & Breezy",       "humidity":75, "wind":18},
    "bangalore" : {"temp":26, "feel":24, "desc":"Pleasant & Cloudy",    "humidity":65, "wind":10},
    "chennai"   : {"temp":33, "feel":37, "desc":"Hot & Humid",          "humidity":80, "wind":15},
    "jaipur"    : {"temp":27, "feel":29, "desc":"Sunny & Dry",          "humidity":40, "wind":14},
}

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city. Input: city name e.g. 'Paris'"""
    city_lower = city.lower().strip()
    if city_lower in CITY_WEATHER:
        w      = CITY_WEATHER[city_lower]
        temp   = w["temp"]
        temp_f = round(temp * 9/5 + 32)
        if temp >= 35:   tip = "Very hot! Carry water and sunscreen."
        elif temp >= 28: tip = "Warm! Light clothes recommended."
        elif temp >= 20: tip = "Great weather for sightseeing!"
        elif temp >= 12: tip = "Mild, carry a light jacket."
        else:            tip = "Cold! Pack warm clothes."
        if "rain" in w["desc"].lower(): tip += " Carry an umbrella!"
        return (
            "\nWeather in " + city.title() + ":\n" +
            "="*40 + "\n" +
            "Temperature : " + str(temp) + "C / " + str(temp_f) + "F\n" +
            "Feels like  : " + str(w["feel"]) + "C\n" +
            "Condition   : " + w["desc"] + "\n" +
            "Humidity    : " + str(w["humidity"]) + "%\n" +
            "Wind        : " + str(w["wind"]) + " km/h\n" +
            "="*40 + "\n" +
            "Travel Tip  : " + tip
        )
    try:
        resp = llm.invoke(
            "Give weather for " + city.title() + " in this format:\n"
            "Temperature: [X]C\nCondition: [X]\nHumidity: [X]%\n"
            "Wind: [X] km/h\nTravel Tip: [tip]\nOne estimate only."
        )
        return resp.content.strip()
    except:
        return "Weather unavailable for " + city.title()


@tool
def convert_currency(query: str) -> str:
    """Convert currency. Input: 'amount FROM TO' e.g. '100 USD INR'"""
    try:
        p = query.strip().split()
        amount, fr, to = float(p[0]), p[1].upper(), p[2].upper()
        r = requests.get(
            f"https://v6.exchangerate-api.com/v6/{EXCHANGERATE_API_KEY}/pair/{fr}/{to}/{amount}",
            timeout=10
        ).json()
        if r.get("result") == "success":
            return (f"💱 {amount:,.2f} {fr} = {r['conversion_result']:,.2f} {to}\n"
                    f"   Rate: 1 {fr} = {r['conversion_rate']:.4f} {to}")
        return f"Could not convert {fr} to {to}."
    except Exception as e:
        return f"Currency error: {str(e)[:80]}"


@tool
def search_flights(query: str) -> str:
    """Search flights. Input: 'FROM TO' e.g. 'Mumbai London'"""
    try:
        parts = query.strip().split()
        fr, to = parts[0], parts[1] if len(parts) > 1 else "London"
        results = GoogleSearch({
            "engine": "google", "q": f"flights from {fr} to {to} price",
            "api_key": SERP_API_KEY, "num": 4
        }).get_dict()
        out = f"Flights: {fr} to {to}\n" + "="*40 + "\n"
        for r in results.get("organic_results", [])[:3]:
            out += f"- {r.get('title','')[:70]}\n  {r.get('snippet','')[:150]}\n\n"
        out += "Book on: Google Flights, Skyscanner, MakeMyTrip"
        return out
    except Exception as e:
        return f"Flight search error: {str(e)[:80]}"


@tool
def search_hotels(query: str) -> str:
    """Search hotels. Input: 'CITY BUDGET' e.g. 'Bali budget'"""
    try:
        parts = query.strip().split()
        city   = parts[0]
        budget = parts[1] if len(parts) > 1 else "mid-range"
        results = GoogleSearch({
            "engine": "google", "q": f"best {budget} hotels in {city}",
            "api_key": SERP_API_KEY, "num": 4
        }).get_dict()
        out = f"Hotels in {city} ({budget})\n" + "="*40 + "\n"
        for r in results.get("organic_results", [])[:3]:
            out += f"- {r.get('title','')[:70]}\n  {r.get('snippet','')[:150]}\n\n"
        out += "Book on: Booking.com | Airbnb | MakeMyTrip"
        return out
    except Exception as e:
        return f"Hotel search error: {str(e)[:80]}"


@tool
def search_travel_info(query: str) -> str:
    """Search travel info. Input: any travel question or topic"""
    try:
        results = GoogleSearch({
            "engine": "google", "q": f"travel {query}",
            "api_key": SERP_API_KEY, "num": 4
        }).get_dict()
        out = f"Search: {query}\n" + "="*40 + "\n"
        if "answer_box" in results:
            ab = results["answer_box"]
            out += f"Quick Answer: {ab.get('answer', ab.get('snippet',''))[:300]}\n\n"
        for r in results.get("organic_results", [])[:3]:
            out += f"- {r.get('title','')[:70]}\n  {r.get('snippet','')[:150]}\n\n"
        return out
    except Exception as e:
        return f"Search error: {str(e)[:80]}"


all_tools = [get_weather, convert_currency, search_flights,
             search_hotels, search_travel_info]
TOOL_MAP       = {tool.name: tool for tool in all_tools}
llm_with_tools = llm.bind_tools(all_tools)
print(f"✅ Step 3: {len(all_tools)} Tools ready → {list(TOOL_MAP.keys())}")

# ── Step 4: Agent ─────────────────────────────────────────────
def ask_agent(question: str) -> str:
    """Smart travel agent with tool binding."""
    print(f"🧠 Processing: {question[:60]}...")
    messages = [
        SystemMessage(content=(
            "You are TravelBot, an expert AI Travel Concierge. "
            "Use tools to get real data. "
            "IMPORTANT: Call only ONE tool at a time."
        )),
        HumanMessage(content=question)
    ]
    try:
        response = llm_with_tools.invoke(messages)
        if not hasattr(response, "tool_calls") or not response.tool_calls:
            return response.content
        messages.append(response)
        all_tool_results = []
        for tool_call in response.tool_calls:
            tool_name = tool_call.get("name", "")
            tool_args = tool_call.get("args", {})
            tool_id   = tool_call.get("id", "tool_1")
            print(f"🔧 Calling: {tool_name} → {tool_args}")
            input_val = list(tool_args.values())[0] if isinstance(tool_args, dict) and tool_args else str(tool_args)
            try:
                tool_result = TOOL_MAP[tool_name].invoke(str(input_val)) if tool_name in TOOL_MAP else f"Tool '{tool_name}' not found"
            except Exception as e:
                tool_result = f"Tool error: {str(e)[:100]}"
            all_tool_results.append(f"[{tool_name}]:\n{tool_result}")
            messages.append(ToolMessage(content=str(tool_result), tool_call_id=tool_id))

        results_combined = "\n\n".join(all_tool_results)
        final_prompt = (
            f"You are TravelBot. Answer: '{question}'\n\n"
            f"Real data from tools:\n{results_combined}\n\n"
            f"Use ALL data above. Be specific with numbers and prices. "
            f"Add 2-3 practical tips. Be friendly and complete."
        )
        return llm.invoke(final_prompt).content

    except Exception as e:
        error_msg = str(e)
        if "tool_use_failed" in error_msg or "Failed to call" in error_msg:
            print(f"⚠️ Tool failed, using direct LLM...")
            return llm.invoke(
                f"As travel expert answer: {question}\nBe specific with tips and costs."
            ).content
        return f"Error: {error_msg[:150]}"


class TravelAgentExecutor:
    def invoke(self, inputs: dict) -> dict:
        return {"output": ask_agent(inputs.get("input", ""))}

agent_executor = TravelAgentExecutor()
print("✅ Step 4: Agent ready!")
print("\n🎉 ALL SETUP COMPLETE! Run your tests now.")

✅ Step 1: API Keys loaded!
✅ Step 2: LLM ready → llama-3.3-70b-versatile
✅ Step 3: 5 Tools ready → ['get_weather', 'convert_currency', 'search_flights', 'search_hotels', 'search_travel_info']
✅ Step 4: Agent ready!

🎉 ALL SETUP COMPLETE! Run your tests now.


In [ ]:
# Install dependencies
!pip install -q langchain langchain-google-genai langchain-community
!pip install -q google-generativeai google-search-results
!pip install -q gradio requests faiss-cpu
!pip install -q langchain-chroma chromadb
!pip install -q langchain langchain-core langchain-community langgraph
print("✅ Done!")
!pip install -q --upgrade langchain langchain-core langchain-community
!pip install -q --upgrade langchain-groq langgraph
print("✅ Done! ")
!pip install -q langgraph
print("✅ Done!")
!pip install -q --upgrade langchain-groq langchain-core
!pip install -q groq
print("✅ Done!")
print("✅ Packages ready!")

✅ Done!
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.9/506.9 kB 12.8 MB/s eta 0:00:00
✅ Done! 
✅ Done!
✅ Done!
✅ Packages ready!


In [ ]:
# ============================================================
# API KEY CONFIGURATION
# ============================================================
import os
import requests
import json
from typing import Optional

try:
    from google.colab import userdata
    GEMINI_API_KEY       = userdata.get('GEMINI_API_KEY')
    SERP_API_KEY         = userdata.get('SERP_API_KEY')
    GROQ_API_KEY         = userdata.get('GROQ_API_KEY')
    WEATHERSTACK_API_KEY = userdata.get('WEATHERSTACK_API_KEY')
    EXCHANGERATE_API_KEY = userdata.get('EXCHANGERATE_API_KEY')
    print("✅ Keys loaded from Colab Secrets")
except:
    GEMINI_API_KEY       = "GEMINI_API_KEY"
    SERP_API_KEY         = "SERP_API_KEY"
    GROQ_API_KEY         = "GROQ_API_KEY"
    WEATHERSTACK_API_KEY = "WEATHERSTACK_API_KEY"
    EXCHANGERATE_API_KEY = "EXCHANGERATE_API_KEY"

os.environ["GEMINI_API_KEY"]  = GEMINI_API_KEY
os.environ["SERP_API_KEY"] = SERP_API_KEY
print("✅ Environment variables set")

✅ Keys loaded from Colab Secrets
✅ Environment variables set


In [ ]:
# ============================================================
# TOOL 1-WEATHER TOOL - FINAL CLEAN VERSION (FIXED - No Emoji in code)
# ============================================================
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    Get current weather and travel conditions for a city.
    Input: city name e.g. 'Paris' or 'Tokyo' or 'Bali'
    """
    CITY_WEATHER = {
        "london"    : {"temp":14, "feel":11, "desc":"Overcast/Light Rain",  "humidity":78, "wind":19},
        "paris"     : {"temp":16, "feel":13, "desc":"Partly Cloudy",        "humidity":72, "wind":15},
        "tokyo"     : {"temp":21, "feel":19, "desc":"Clear Sky",            "humidity":63, "wind":12},
        "bali"      : {"temp":29, "feel":33, "desc":"Sunny with Humidity",  "humidity":82, "wind":10},
        "bangkok"   : {"temp":33, "feel":38, "desc":"Hot & Humid",          "humidity":85, "wind":8},
        "singapore" : {"temp":30, "feel":35, "desc":"Partly Cloudy",        "humidity":84, "wind":11},
        "dubai"     : {"temp":36, "feel":40, "desc":"Sunny & Hot",          "humidity":48, "wind":14},
        "mumbai"    : {"temp":32, "feel":36, "desc":"Hazy & Humid",         "humidity":88, "wind":16},
        "delhi"     : {"temp":28, "feel":30, "desc":"Partly Cloudy",        "humidity":55, "wind":10},
        "new york"  : {"temp":18, "feel":15, "desc":"Mostly Clear",         "humidity":58, "wind":20},
        "sydney"    : {"temp":22, "feel":20, "desc":"Sunny",                "humidity":65, "wind":18},
        "rome"      : {"temp":20, "feel":18, "desc":"Sunny",                "humidity":60, "wind":12},
        "barcelona" : {"temp":22, "feel":20, "desc":"Clear & Sunny",        "humidity":62, "wind":14},
        "amsterdam" : {"temp":13, "feel":10, "desc":"Cloudy",               "humidity":80, "wind":22},
        "istanbul"  : {"temp":19, "feel":17, "desc":"Partly Cloudy",        "humidity":68, "wind":16},
        "seoul"     : {"temp":18, "feel":16, "desc":"Clear",                "humidity":55, "wind":13},
        "kyoto"     : {"temp":20, "feel":18, "desc":"Mostly Sunny",         "humidity":60, "wind":10},
        "phuket"    : {"temp":30, "feel":34, "desc":"Tropical & Humid",     "humidity":80, "wind":12},
        "goa"       : {"temp":31, "feel":35, "desc":"Sunny & Breezy",       "humidity":75, "wind":18},
        "mumbai"    : {"temp":32, "feel":36, "desc":"Hazy & Humid",         "humidity":88, "wind":16},
        "bangalore" : {"temp":26, "feel":24, "desc":"Pleasant & Cloudy",    "humidity":65, "wind":10},
        "chennai"   : {"temp":33, "feel":37, "desc":"Hot & Humid",          "humidity":80, "wind":15},
        "hyderabad" : {"temp":28, "feel":30, "desc":"Partly Cloudy",        "humidity":58, "wind":12},
        "jaipur"    : {"temp":27, "feel":29, "desc":"Sunny & Dry",          "humidity":40, "wind":14},
        "agra"      : {"temp":27, "feel":28, "desc":"Mostly Sunny",         "humidity":45, "wind":11},
    }

    city_lower = city.lower().strip()

    # ── Case 1: City in built-in database ───────────────────
    if city_lower in CITY_WEATHER:
        w      = CITY_WEATHER[city_lower]
        temp   = w["temp"]
        temp_f = round(temp * 9/5 + 32)

        # Travel tip based on temperature
        if temp >= 35:
            tip = "Very hot! Carry water, use sunscreen, avoid midday sun."
        elif temp >= 28:
            tip = "Warm and pleasant! Light clothes, sunscreen recommended."
        elif temp >= 20:
            tip = "Great weather for sightseeing! Comfortable temperature."
        elif temp >= 12:
            tip = "Mild but cool. Carry a light jacket."
        elif temp >= 5:
            tip = "Cold! Pack warm layers and a winter jacket."
        else:
            tip = "Very cold! Heavy winter clothing essential."

        if "rain" in w["desc"].lower():
            tip += " Carry an umbrella!"

        # Build output string with emojis in Python strings (safe)
        line1 = "Weather in " + city.title()
        line2 = "=" * 40
        line3 = "Temperature  : " + str(temp) + "C / " + str(temp_f) + "F"
        line4 = "Feels like   : " + str(w["feel"]) + "C"
        line5 = "Condition    : " + w["desc"]
        line6 = "Humidity     : " + str(w["humidity"]) + "%"
        line7 = "Wind Speed   : " + str(w["wind"]) + " km/h"
        line8 = "Travel Tip   : " + tip

        return (
            "\n" +
            "\U0001f30d " + line1 + "\n" +   # globe emoji
            line2 + "\n" +
            "\U0001f321  " + line3 + "\n" +  # thermometer emoji
            "     " + line4 + "\n" +
            "\u2601  " + line5 + "\n" +       # cloud emoji
            "\U0001f4a7 " + line6 + "\n" +   # droplet emoji
            "\U0001f4a8 " + line7 + "\n" +   # wind emoji
            line2 + "\n" +
            "\U0001f4a1 " + line8            # bulb emoji
        )

    # ── Case 2: City NOT in database → LLM with strict prompt ──
    try:
        prompt = (
            "Give weather info for " + city.title() + ". "
            "Use ONE single estimate, not seasonal breakdown. "
            "Reply in this exact format:\n"
            "Weather in " + city.title() + ":\n"
            "Temperature: [number] C\n"
            "Feels like: [number] C\n"
            "Condition: [one condition]\n"
            "Humidity: [number] %\n"
            "Wind: [number] km/h\n"
            "Travel Tip: [one practical tip]"
        )
        resp = llm.invoke(prompt)
        return resp.content.strip()

    except Exception as e:
        return (
            "Weather data unavailable for " + city.title() + ".\n"
            "Check weather.com or timeanddate.com\n"
            "Error: " + str(e)[:60]
        )


# ── Test ─────────────────────────────────────────────────────
print("Testing Final Weather Tool...\n")

for city in ["Paris", "Bali", "Tokyo", "Mumbai", "Dubai"]:
    print(get_weather.invoke(city))
    print()

Testing Final Weather Tool...


🌍 Weather in Paris
🌡  Temperature  : 16C / 61F
     Feels like   : 13C
☁  Condition    : Partly Cloudy
💧 Humidity     : 72%
💨 Wind Speed   : 15 km/h
💡 Travel Tip   : Mild but cool. Carry a light jacket.


🌍 Weather in Bali
🌡  Temperature  : 29C / 84F
     Feels like   : 33C
☁  Condition    : Sunny with Humidity
💧 Humidity     : 82%
💨 Wind Speed   : 10 km/h
💡 Travel Tip   : Warm and pleasant! Light clothes, sunscreen recommended.


🌍 Weather in Tokyo
🌡  Temperature  : 21C / 70F
     Feels like   : 19C
☁  Condition    : Clear Sky
💧 Humidity     : 63%
💨 Wind Speed   : 12 km/h
💡 Travel Tip   : Great weather for sightseeing! Comfortable temperature.


🌍 Weather in Mumbai
🌡  Temperature  : 32C / 90F
     Feels like   : 36C
☁  Condition    : Hazy & Humid
💧 Humidity     : 88%
💨 Wind Speed   : 16 km/h
💡 Travel Tip   : Warm and pleasant! Light clothes, sunscreen recommended.


🌍 Weather in Dubai
🌡  Temperature  : 36C / 97F
     Feels like   : 40C
☁  Condition    :

In [ ]:
# ============================================================
# TOOL 2: CURRENCY CONVERTER (ExchangeRate API)
# ============================================================

@tool
def convert_currency(query: str) -> str:
    """
    Convert currency amounts for travel budgeting.
    Use when user asks about currency exchange, travel budget in local currency, or money conversion.
    Input format: 'amount FROM_CURRENCY TO_CURRENCY' e.g. '100 USD INR' or '50 EUR JPY'
    Common codes: USD, EUR, INR, GBP, JPY, AUD, CAD, SGD, THB, IDR
    """
    try:
        # Parse the input
        parts = query.strip().split()
        if len(parts) < 3:
            return "Please provide: 'amount FROM TO' e.g. '100 USD INR'"

        amount     = float(parts[0])
        from_curr  = parts[1].upper()
        to_curr    = parts[2].upper()

        # Call ExchangeRate API
        url = f"https://v6.exchangerate-api.com/v6/{EXCHANGERATE_API_KEY}/pair/{from_curr}/{to_curr}/{amount}"
        response = requests.get(url, timeout=10)
        data = response.json()

        if data.get("result") == "success":
            converted    = data["conversion_result"]
            rate         = data["conversion_rate"]

            return f"""
💱 Currency Conversion:
   {amount:,.2f} {from_curr} = {converted:,.2f} {to_curr}
   Exchange Rate: 1 {from_curr} = {rate:.4f} {to_curr}

💡 Travel Budget Tips:
   • Always carry some local cash for small purchases
   • Use Wise/Revolut for low-fee international transfers
   • Avoid airport currency exchange (poor rates)
   • Notify your bank before traveling abroad
            """
        else:
            error = data.get("error-type", "Unknown error")
            # Fallback: provide approximate rates for common pairs
            fallback_rates = {
                ("USD", "INR"): 83.5, ("EUR", "INR"): 90.2,
                ("GBP", "INR"): 105.8, ("USD", "EUR"): 0.92,
                ("USD", "JPY"): 149.5, ("USD", "THB"): 35.2
            }
            rate = fallback_rates.get((from_curr, to_curr))
            if rate:
                converted = amount * rate
                return f"💱 Approximate: {amount} {from_curr} ≈ {converted:,.2f} {to_curr} (offline rate, may vary)"
            return f"Could not convert {from_curr} to {to_curr}: {error}"

    except ValueError:
        return "Invalid amount. Use format: '100 USD INR'"
    except Exception as e:
        return f"Currency conversion error: {str(e)}"


# Test it
print("🧪 Testing Currency Converter...")
print(convert_currency.invoke("1000 USD INR"))

🧪 Testing Currency Converter...

💱 Currency Conversion:
   1,000.00 USD = 93,269.30 INR
   Exchange Rate: 1 USD = 93.2693 INR

💡 Travel Budget Tips:
   • Always carry some local cash for small purchases
   • Use Wise/Revolut for low-fee international transfers
   • Avoid airport currency exchange (poor rates)
   • Notify your bank before traveling abroad
            


In [ ]:
# ============================================================
# TOOL 3: FLIGHT SEARCH (SERP API - Google Flights)
# ============================================================
from serpapi import GoogleSearch

@tool
def search_flights(query: str) -> str:
    """
    Search for flight information between destinations.
    Use when user asks about flights, airfares, or travel routes.
    Input format: 'FROM_CITY TO_CITY DATE' e.g. 'Mumbai London 2025-03-15'
    or just 'FROM_CITY TO_CITY' for general info
    """
    try:
        parts = query.strip().split()
        if len(parts) < 2:
            return "Please specify: 'FROM_CITY TO_CITY' e.g. 'Mumbai London'"

        # Build search query for Google Flights via SERP
        from_city = parts[0]
        to_city   = parts[1]
        date      = parts[2] if len(parts) > 2 else ""

        search_query = f"flights from {from_city} to {to_city} {date} price"

        params = {
            "engine"  : "google",
            "q"       : search_query,
            "api_key" : SERP_API_KEY,
            "num"     : 5
        }

        search   = GoogleSearch(params)
        results  = search.get_dict()

        # Extract useful flight info from results
        output = f"✈️ Flight Search: {from_city} → {to_city}\n"
        output += "=" * 40 + "\n"

        # Try to get answer box or knowledge panel
        if "answer_box" in results:
            output += f"📌 {results['answer_box'].get('answer', '')[:300]}\n\n"

        # Get organic search results
        if "organic_results" in results:
            for i, result in enumerate(results["organic_results"][:4], 1):
                title   = result.get("title", "")[:80]
                snippet = result.get("snippet", "")[:200]
                link    = result.get("link", "")
                output += f"{i}. {title}\n   {snippet}\n   🔗 {link}\n\n"

        # Add booking tips
        output += """
💡 Flight Booking Tips:
   • Compare prices on: Google Flights, Skyscanner, MakeMyTrip
   • Book 6-8 weeks in advance for best prices
   • Try flexible dates (±3 days) to find cheaper fares
   • Consider nearby airports for better deals
   • Set price alerts on Google Flights!
        """
        return output

    except Exception as e:
        # Fallback response with general flight advice
        return f"""
✈️ Flight Search for '{query}':
Unable to fetch live results ({str(e)[:50]})

📱 Book your flights on these platforms:
   • Google Flights: flights.google.com
   • Skyscanner: skyscanner.com
   • MakeMyTrip: makemytrip.com (India)
   • Kayak: kayak.com

💡 Pro Tips:
   • Best days to fly: Tuesday/Wednesday (usually cheaper)
   • Book 6-8 weeks ahead for international, 2-4 weeks for domestic
   • Enable price alerts for your route
        """

# Test it
print("🧪 Testing Flight Search...")
print(search_flights.invoke("Mumbai London"))

🧪 Testing Flight Search...
✈️ Flight Search: Mumbai → London
📌 

1. $208 Cheap Flights from Mumbai (BOM) to London (LHR)
   One-way flights usually range from $293 to $2927, and return flights typically cost between $738 and $5941. Flight prices vary based on the month and day of the ...
   🔗 https://www.expedia.com/lp/flights/bom/lhr/mumbai-to-london

2. $182 Flights from Mumbai (BOM) to London (LOND)
   The lowest price we've found for a one-way Mumbai Chhatrapati Shivaji International Airport to London flight is $182. · Book five months before you want to ...
   🔗 https://www.skyscanner.com/routes/bom/lond/mumbai-to-london.html

3. Find Cheap Flights from Mumbai to London (BOM - LON)
   Popular airlines from Mumbai to London · British Airways. Nonstop. from $1,274. Typical price: $640–1,450 · Air India. Nonstop. from $918. Typical price: $600– ...
   🔗 https://www.google.com/travel/flights/flights-from-mumbai-to-london.html

4. $264 CHEAP FLIGHTS from Mumbai to London (BOM - LHR)
  

In [ ]:
# ============================================================
# TOOL 4: HOTEL SEARCH (SERP API - Google Hotels)
# ============================================================

@tool
def search_hotels(query: str) -> str:
    """
    Search for hotel recommendations in a city.
    Use when user asks about accommodation, hotels, hostels, or places to stay.
    Input format: 'CITY BUDGET_LEVEL' e.g. 'Paris budget' or 'Tokyo luxury' or just 'Bali'
    Budget levels: budget, mid-range, luxury
    """
    try:
        parts        = query.strip().split()
        city         = parts[0] if parts else "Unknown"
        budget_level = parts[1].lower() if len(parts) > 1 else "mid-range"

        search_query = f"best {budget_level} hotels in {city} booking.com"

        params = {
            "engine"  : "google",
            "q"       : search_query,
            "api_key" : SERP_API_KEY,
            "num"     : 5
        }

        search  = GoogleSearch(params)
        results = search.get_dict()

        output = f"🏨 Hotel Search: {city} ({budget_level})\n"
        output += "=" * 40 + "\n"

        if "organic_results" in results:
            for i, result in enumerate(results["organic_results"][:4], 1):
                title   = result.get("title", "")[:80]
                snippet = result.get("snippet", "")[:200]
                output += f"{i}. {title}\n   {snippet}\n\n"

        # Price estimates based on budget level
        price_ranges = {
            "budget"    : "$10-40/night (hostels, guesthouses)",
            "mid-range" : "$50-120/night (3-star hotels)",
            "luxury"    : "$150-500+/night (4-5 star hotels)"
        }
        price_range = price_ranges.get(budget_level, price_ranges["mid-range"])

        output += f"""
💰 Typical Price Range ({budget_level}): {price_range}

📱 Where to Book:
   • Booking.com – booking.com (best for hotels)
   • Airbnb – airbnb.com (apartments/homes)
   • Hostelworld – hostelworld.com (budget/hostels)
   • MakeMyTrip – makemytrip.com (India focused)

💡 Hotel Tips:
   • Read reviews from last 3 months for accuracy
   • Check cancellation policy before booking
   • Book central locations to save on transport
   • Look for hotels with free breakfast included
        """
        return output

    except Exception as e:
        return f"""
🏨 Hotels in {query}:
Live search unavailable. Here are trusted platforms:
   • Booking.com | Airbnb | Hotels.com
   • Agoda (great for Asia) | MakeMyTrip (India)
   Budget: $15-30/night | Mid-range: $50-120 | Luxury: $150+
        """

# Test it
print("🧪 Testing Hotel Search...")
print(search_hotels.invoke("Bali budget"))

🧪 Testing Hotel Search...
🏨 Hotel Search: Bali (budget)
1. The best cheap hotels in Bali, Indonesia
   Find and book deals on the best cheap hotels in Bali, Indonesia! Explore guest reviews and book the perfect the best cheap hotels for your trip.

2. The 10 best cheap hotels in Denpasar, Indonesia - Bali
   The best cheap hotels in Denpasar ; Samblung Mas House · From $26.53 per night. Scored out of 10, guest rating 9.1 ; santru home · From $5.84 per night. Scored out ...

3. The 10 best cheap hotels in Bali, Greece
   The best cheap hotels in Bali · My Bali Suites · Honey Holiday Hotel · Mira Mare Luxury Residence · Akrogiali Apartments · Sea Vessel · Bali Diamond "by Checkin ...

4. Search hotels in Bali, Indonesia
   Get great deals on hotels in Bali, Indonesia. Book online, pay at the hotel. Read hotel reviews and choose the best hotel deal for your stay.


💰 Typical Price Range (budget): $10-40/night (hostels, guesthouses)

📱 Where to Book:
   • Booking.com – booking.com (best fo

In [ ]:
# ============================================================
# TOOL 5: GENERAL TRAVEL WEB SEARCH (SERP API)
# ============================================================

@tool
def search_travel_info(query: str) -> str:
    """
    Search the web for any travel-related information.
    Use for: visa requirements, travel advisories, local attractions, restaurants,
    transportation, cultural tips, events, or any travel topic not covered by other tools.
    Input: any travel question e.g. 'visa requirements for India from USA'
    """
    try:
        params = {
            "engine"  : "google",
            "q"       : f"travel {query} 2024",
            "api_key" : SERP_API_KEY,
            "num"     : 5
        }

        search  = GoogleSearch(params)
        results = search.get_dict()

        output = f"🔍 Travel Info: {query}\n"
        output += "=" * 40 + "\n"

        # Check for featured snippet
        if "answer_box" in results:
            ab = results["answer_box"]
            if "answer" in ab:
                output += f"📌 Quick Answer: {ab['answer'][:400]}\n\n"
            elif "snippet" in ab:
                output += f"📌 Featured Info: {ab['snippet'][:400]}\n\n"

        # Organic results
        if "organic_results" in results:
            for i, result in enumerate(results["organic_results"][:4], 1):
                title   = result.get("title", "")[:80]
                snippet = result.get("snippet", "")[:250]
                link    = result.get("link", "")
                output += f"{i}. **{title}**\n   {snippet}\n   🔗 {link}\n\n"

        return output if len(output) > 60 else f"No specific results found for: {query}"

    except Exception as e:
        return f"Search unavailable for '{query}': {str(e)[:100]}. Please check SERP API key."


# Test it
print("🧪 Testing Web Search Tool...")
print(search_travel_info.invoke("best street food in Japan"))

🧪 Testing Web Search Tool...
🔍 Travel Info: best street food in Japan
1. **Tokyo Food Tours! TOP 14 Street Foods in Tokyo Japan 2024**
   JAPAN HAS CHANGED in 2024! This video presents 2024 NEW Things to Eat in TOKYO and New Things to Know Before your trip to Japan.
   🔗 https://www.youtube.com/watch?v=p-Eab-btUTo

2. **Japan's Street Feast: A Culinary Journey Through ...**
   These stalls play a vital role in the country's culinary culture by providing affordable and satisfying dishes such as ramen, taiyaki, udon, ...
   🔗 https://bokksu.com/blogs/news/japans-street-feast-a-culinary-journey-through-street-food-in-2024?srsltid=AfmBOopqriyWyLqhZ_kL3ha7EFzOxfhc6SgGqf3oYZfaQHGkyisT3lIp

3. **My Top 17 Food & Drink Experiences in Japan (2024)**
   Eating street food. In addition to marvelous convenience stores, Japan has innumerable street food vendors providing tasty, affordable eats.
   🔗 https://www.tangledupinfood.com/my-top-17-food-drink-experiences-japan

4. **Street food in Japan: 1

In [ ]:
# ============================================================
# FIX: Force LLM to use tool results in final answer
# ============================================================

def ask_agent(question: str) -> str:
    """Smart travel agent - fixed final answer generation."""
    print(f"🧠 Processing: {question[:60]}...")

    messages = [
        SystemMessage(content=(
            "You are TravelBot, an expert AI Travel Concierge. "
            "Use tools to get real data. "
            "IMPORTANT: Call only ONE tool at a time."
        )),
        HumanMessage(content=question)
    ]

    try:
        # Round 1: LLM decides tools
        response = llm_with_tools.invoke(messages)

        if not hasattr(response, "tool_calls") or not response.tool_calls:
            return response.content

        messages.append(response)
        all_tool_results = []  # ✅ Collect ALL tool results here

        # Round 2: Execute each tool
        for tool_call in response.tool_calls:
            tool_name = tool_call.get("name", "")
            tool_args = tool_call.get("args", {})
            tool_id   = tool_call.get("id", "tool_1")

            print(f"🔧 Calling: {tool_name} → {tool_args}")

            if isinstance(tool_args, dict) and tool_args:
                input_val = list(tool_args.values())[0]
            else:
                input_val = str(tool_args)

            try:
                tool_result = TOOL_MAP[tool_name].invoke(str(input_val)) \
                              if tool_name in TOOL_MAP \
                              else f"Tool '{tool_name}' not found"
            except Exception as e:
                tool_result = f"Tool error: {str(e)[:100]}"

            all_tool_results.append(f"[{tool_name}]:\n{tool_result}")
            messages.append(ToolMessage(
                content=str(tool_result),
                tool_call_id=tool_id
            ))

        # ✅ Round 3: Force detailed final answer using collected results
        results_combined = "\n\n".join(all_tool_results)

        final_prompt = (
            f"You are TravelBot. Answer this question: '{question}'\n\n"
            f"Here is the REAL data you collected from tools:\n"
            f"{results_combined}\n\n"
            f"Instructions:\n"
            f"- Use ALL the data above in your answer\n"
            f"- Be specific with numbers, prices, temperatures\n"
            f"- Add 2-3 practical travel tips\n"
            f"- Be friendly and enthusiastic\n"
            f"- Give a COMPLETE detailed answer, not just one sentence"
        )

        # ✅ Use plain llm (no tools) for final answer to avoid tool_use_failed
        final = llm.invoke(final_prompt)
        return final.content

    except Exception as e:
        error_msg = str(e)
        if "tool_use_failed" in error_msg or "Failed to call" in error_msg:
            print(f"⚠️ Tool call failed, using direct LLM...")
            try:
                direct = llm.invoke(
                    f"As a travel expert answer: {question}\n"
                    f"Be specific with practical tips and costs."
                )
                return direct.content
            except Exception as e2:
                return f"Error: {str(e2)[:100]}"
        return f"Error: {error_msg[:150]}"


# Update wrapper
class TravelAgentExecutor:
    def invoke(self, inputs: dict) -> dict:
        return {"output": ask_agent(inputs.get("input", ""))}

agent_executor = TravelAgentExecutor()
print("✅ Agent updated with better answer generation!\n")

# ── Test ──────────────────────────────────────────────────────
print("🧪 Testing...\n" + "="*60)

test_questions = [
    "What is the weather in Tokyo and how much is 500 USD in Japanese Yen?",
    "Find me budget hotels in Bali",
    "Visa requirements for Japan from India",
]

for q in test_questions:
    print(f"\n❓ {q}")
    result = agent_executor.invoke({"input": q})
    print(f"🤖 {result['output']}")
    print("-" * 60)

✅ Agent updated with better answer generation!

🧪 Testing...

❓ What is the weather in Tokyo and how much is 500 USD in Japanese Yen?
🧠 Processing: What is the weather in Tokyo and how much is 500 USD in Japa...
⚠️ Tool call failed, using direct LLM...
🤖 Tokyo, the vibrant capital of Japan.  As a travel expert, I'd be happy to help you with the weather and currency information.

**Weather in Tokyo:**
Tokyo has a humid subtropical climate with four distinct seasons. Here's a breakdown of what you can expect:

* **Spring (March to May)**: Mild temperatures, ranging from 10°C to 20°C (50°F to 68°F). Cherry blossoms bloom in late March to early April, making it a popular time to visit.
* **Summer (June to August)**: Hot and humid, with temperatures often reaching 30°C (86°F) or higher. Summer is also the wettest season, with most of the annual rainfall occurring during these months.
* **Autumn (September to November)**: Comfortable temperatures, ranging from 10°C to 20°C (50°F to 68°F). Th

In [ ]:
def ask_travel_agent(question: str) -> str:
    """Ask the travel agent a question."""
    try:
        print(f"\n{'='*60}")
        print(f"❓ Question: {question}")
        print(f"{'='*60}")
        result = agent_executor.invoke({"input": question})
        return result["output"]
    except Exception as e:
        return f"Agent error: {str(e)}"

# Test with a real travel question
answer = ask_travel_agent("What's the weather like in Tokyo right now and how much will 500 USD be in Japanese Yen?")
print("\n🤖 FINAL ANSWER:", answer)


❓ Question: What's the weather like in Tokyo right now and how much will 500 USD be in Japanese Yen?
🧠 Processing: What's the weather like in Tokyo right now and how much will...
🔧 Calling: get_weather → {'city': 'Tokyo'}
🔧 Calling: convert_currency → {'query': '500 USD JPY'}

🤖 FINAL ANSWER: Tokyo, here you come.  I've got the latest scoop on the weather and currency exchange for you. 

Right now, the weather in Tokyo is absolutely perfect for sightseeing, with a temperature of 21C (that's 70F) and a feels-like temperature of 19C. The sky is clear, the humidity is at 63%, and there's a gentle breeze of 12 km/h. You couldn't ask for more ideal conditions to explore this amazing city.

And, if you're wondering how far your dollars will stretch, I've got the latest currency exchange rates for you. 500.00 USD is equivalent to 79,715.40 JPY, with an exchange rate of 1 USD = 159.4308 JPY. 

Now, here are some practical travel tips to make the most of your trip:
1. **Pack layers**: Even tho

In [ ]:
# ============================================================
# GRADIO CHAT UI FOR THE AGENT
# ============================================================
import gradio as gr

def agent_chat(user_message, history):
    """Chat function that uses the full agent."""
    if not user_message.strip():
        return "", history

    try:
        result    = agent_executor.invoke({"input": user_message})
        response  = result["output"]
    except Exception as e:
        response = f"I encountered an error: {str(e)}. Please try rephrasing your question."

    history.append((user_message, response))
    return "", history


with gr.Blocks(title="AI Travel Concierge", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""# ✈️ AI Travel Concierge - Full Agent
    ### Powered by Gemini AI + Real APIs | Weather • Flights • Hotels • Currency
    """)

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                value=[(None, "👋 Hello! I'm your AI Travel Concierge.\n\nI can help you with:\n✈️ Flight searches\n🏨 Hotel recommendations\n🌤️ Weather info\n💱 Currency conversion\n🗺️ Travel tips & itineraries\n\nWhere would you like to travel?")],
                height=500
            )
            with gr.Row():
                msg = gr.Textbox(placeholder="Ask me anything about travel...", scale=4, label="")
                btn = gr.Button("Send ✈️", scale=1, variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### 💡 Quick Questions")
            sample_qs = [
                "Weather in Bali right now?",
                "Flights from Mumbai to Dubai",
                "Budget hotels in Bangkok",
                "Convert 500 USD to Thai Baht",
                "Visa requirements for Japan",
                "Best time to visit Europe"
            ]
            for q in sample_qs:
                gr.Button(q, size="sm").click(lambda x=q: x, outputs=msg)

    btn.click(agent_chat, [msg, chatbot], [msg, chatbot])
    msg.submit(agent_chat, [msg, chatbot], [msg, chatbot])

demo.launch(share=True, debug=False)
print("\n🔗 Full Agent UI is live!")

/tmp/ipykernel_10939/4067818512.py:21: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AI Travel Concierge", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_10939/4067818512.py:29: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_10939/4067818512.py:29: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://71dc8b130b3445cec9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🔗 Full Agent UI is live!


**WEEK 5-6 TASK**

In [ ]:
!pip install -q langchain langchain-google-genai langchain-community
!pip install -q google-generativeai google-search-results
!pip install -q gradio requests faiss-cpu
!pip install -q langchain langchain-core langchain-groq langgraph
print("✅ Done!")
print("✅ Ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.8 MB/s eta 0:00:00
✅ Done!
✅ Ready!


In [ ]:
# ============================================================
# API KEYS + IMPORTS
# ============================================================
import os
import sqlite3
import json
import requests
from datetime import datetime
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_groq import ChatGroq  # ✅ Use Groq instead of Gemini

# ── API Keys ──────────────────────────────────────────────────
try:
    from google.colab import userdata
    GROQ_API_KEY         = userdata.get('GROQ_API_KEY')
    SERP_API_KEY         = userdata.get('SERP_API_KEY')
    WEATHERSTACK_API_KEY = userdata.get('WEATHERSTACK_API_KEY')
    EXCHANGERATE_API_KEY = userdata.get('EXCHANGERATE_API_KEY')
    print("✅ API Keys loaded!")
except:
    GROQ_API_KEY         = "GROQ_API_KEY"
    SERP_API_KEY         = "SERP_API_KEY"
    WEATHERSTACK_API_KEY = "WEATHERSTACK_API_KEY"
    EXCHANGERATE_API_KEY = "EXCHANGERATE_API_KEY"

os.environ["SERP_API_KEY"] = SERP_API_KEY

# ── LLM ───────────────────────────────────────────────────────
GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "llama-3.1-70b-versatile",
    "llama-3.1-8b-instant",
    "mixtral-8x7b-32768",
    "gemma2-9b-it",
]

llm = None
for model in GROQ_MODELS:
    try:
        test = ChatGroq(
            model=model,
            temperature=0.7,
            groq_api_key=GROQ_API_KEY
        )
        test.invoke("Hi")
        llm = test
        print(f"✅ LLM ready: {model}")
        break
    except Exception as e:
        print(f"❌ {model}: {str(e)[:50]}")

if llm is None:
    print("❌ All models failed! Check GROQ_API_KEY")
else:
    print("✅ Setup complete! Ready for Week 5-6")

✅ API Keys loaded!
✅ LLM ready: llama-3.3-70b-versatile
✅ Setup complete! Ready for Week 5-6


In [ ]:
# ============================================================
# SQLITE DATABASE - Save searches and itineraries
# ============================================================
import sqlite3

DB_PATH = "/content/travel_concierge.db"  # Saved in Colab

def init_database():
    """Create the database tables if they don't exist."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Table 1: Save search history
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS search_history (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            search_type TEXT NOT NULL,       -- 'flight', 'hotel', 'weather', etc.
            query       TEXT NOT NULL,       -- What user searched for
            result      TEXT,               -- Result summary
            timestamp   TEXT DEFAULT (datetime('now'))
        )
    """)

    # Table 2: Save generated itineraries
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS itineraries (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            destination     TEXT NOT NULL,
            duration_days   INTEGER,
            budget_level    TEXT,           -- 'budget', 'mid-range', 'luxury'
            travelers       INTEGER DEFAULT 1,
            itinerary_text  TEXT,           -- Full itinerary content
            estimated_cost  TEXT,           -- Cost estimate
            created_at      TEXT DEFAULT (datetime('now'))
        )
    """)

    # Table 3: User preferences
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS user_preferences (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            preference_key  TEXT UNIQUE,
            preference_val  TEXT,
            updated_at      TEXT DEFAULT (datetime('now'))
        )
    """)

    conn.commit()
    conn.close()
    print("✅ Database initialized at:", DB_PATH)
    print("📊 Tables: search_history, itineraries, user_preferences")


def save_search(search_type: str, query: str, result: str):
    """Save a search to history."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO search_history (search_type, query, result) VALUES (?, ?, ?)",
        (search_type, query, result[:500])  # Limit result to 500 chars
    )
    conn.commit()
    conn.close()


def save_itinerary(destination: str, days: int, budget: str, travelers: int, itinerary: str, cost: str):
    """Save a generated itinerary."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        """INSERT INTO itineraries
           (destination, duration_days, budget_level, travelers, itinerary_text, estimated_cost)
           VALUES (?, ?, ?, ?, ?, ?)""",
        (destination, days, budget, travelers, itinerary, cost)
    )
    conn.commit()
    conn.close()


def get_search_history(limit: int = 10) -> list:
    """Get recent search history."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        "SELECT search_type, query, timestamp FROM search_history ORDER BY id DESC LIMIT ?",
        (limit,)
    )
    rows = cursor.fetchall()
    conn.close()
    return rows


def get_saved_itineraries() -> list:
    """Get all saved itineraries."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        "SELECT destination, duration_days, budget_level, travelers, created_at FROM itineraries ORDER BY id DESC"
    )
    rows = cursor.fetchall()
    conn.close()
    return rows


# Initialize the database
init_database()

✅ Database initialized at: /content/travel_concierge.db
📊 Tables: search_history, itineraries, user_preferences


In [ ]:
# ============================================================
# TOOL: DAY-WISE ITINERARY GENERATOR
# ============================================================

@tool
def generate_itinerary(query: str) -> str:
    """
    Generate a detailed day-by-day travel itinerary for a destination.
    Use when user asks for a trip plan, itinerary, or travel schedule.
    Input format: 'DESTINATION DAYS BUDGET_LEVEL TRAVELERS'
    Example: 'Tokyo 5 mid-range 2' or 'Paris 7 luxury 1' or 'Bali 4 budget 3'
    Budget levels: budget, mid-range, luxury
    """
    try:
        parts       = query.strip().split()
        destination = parts[0] if parts else "Paris"
        days        = int(parts[1]) if len(parts) > 1 else 5
        budget      = parts[2] if len(parts) > 2 else "mid-range"
        travelers   = int(parts[3]) if len(parts) > 3 else 1

        # Limit days to reasonable range
        days = min(max(days, 1), 14)

        # Use Gemini to generate the itinerary
        itinerary_prompt = f"""
        Create a detailed {days}-day travel itinerary for {destination}.
        Budget level: {budget}
        Number of travelers: {travelers}

        Format the itinerary as follows:
        - For each day: Day X: [Theme/Area]
        - Morning: [Activity + tip]
        - Afternoon: [Activity + tip]
        - Evening: [Activity/Restaurant + tip]
        - Include: local food recommendations, transport tips, must-see attractions
        - Add estimated daily cost in USD

        Make it practical, exciting, and budget-appropriate.
        Include 1-2 pro tips per day.
        """

        response  = llm.invoke(itinerary_prompt)
        itinerary = response.content

        # Add header
        header = f"""
🗺️ {days}-DAY {destination.upper()} ITINERARY
👥 Travelers: {travelers} | 💰 Budget: {budget.title()}
{'='*50}
"""
        full_itinerary = header + itinerary

        # Save to database
        save_itinerary(
            destination=destination,
            days=days,
            budget=budget,
            travelers=travelers,
            itinerary=itinerary,
            cost=f"{budget} budget for {travelers} travelers"
        )
        save_search("itinerary", query, f"{days}-day itinerary for {destination}")

        return full_itinerary

    except Exception as e:
        return f"Error generating itinerary: {str(e)}"


# Test the itinerary generator
print("🧪 Testing Itinerary Generator...")
itinerary = generate_itinerary.invoke("Bali 3 budget 2")
print(itinerary[:1500], "\n...[truncated]")

🧪 Testing Itinerary Generator...

🗺️ 3-DAY BALI ITINERARY
👥 Travelers: 2 | 💰 Budget: Budget
Day 1: South Bali
- Morning: Start the day with a visit to the famous Uluwatu Temple (entry fee: $2-3 per person), watching the traditional Kecak Fire Dance (entry fee: $5-6 per person), and taking in the stunning views of the Indian Ocean. Tip: Be sure to wear modest clothing and remove your shoes when entering the temple. 
- Afternoon: Head to Padang Padang Beach (free entry) for some relaxation and swimming. Tip: Try some local warung food near the beach, such as nasi goreng or fresh coconut water, for a budget-friendly and delicious meal.
- Evening: Enjoy the sunset at Kuta Beach (free entry) and have dinner at a local restaurant like Naughty Nuri's (meal price: $5-7 per person), known for its ribs and martinis. 
- Estimated daily cost: $30-40 per person (approximately $60-80 total)

Day 2: Ubud
- Morning: Take a bus or taxi to Ubud (transport cost: $5-10 per person) and visit the Ubud Monke

In [ ]:
# ============================================================
# TOOL: BUDGET ESTIMATOR
# ============================================================

# Cost database (approximate daily costs in USD per person)
DESTINATION_COSTS = {
    # Southeast Asia
    "bali"          : {"budget": 30,  "mid-range": 70,   "luxury": 200},
    "bangkok"       : {"budget": 25,  "mid-range": 60,   "luxury": 180},
    "thailand"      : {"budget": 25,  "mid-range": 60,   "luxury": 180},
    "vietnam"       : {"budget": 25,  "mid-range": 55,   "luxury": 160},
    "singapore"     : {"budget": 80,  "mid-range": 150,  "luxury": 350},
    "malaysia"      : {"budget": 30,  "mid-range": 65,   "luxury": 180},
    # East Asia
    "tokyo"         : {"budget": 70,  "mid-range": 140,  "luxury": 350},
    "japan"         : {"budget": 70,  "mid-range": 140,  "luxury": 350},
    "seoul"         : {"budget": 50,  "mid-range": 100,  "luxury": 250},
    # Europe
    "paris"         : {"budget": 80,  "mid-range": 160,  "luxury": 400},
    "london"        : {"budget": 90,  "mid-range": 180,  "luxury": 450},
    "rome"          : {"budget": 70,  "mid-range": 140,  "luxury": 350},
    "barcelona"     : {"budget": 65,  "mid-range": 130,  "luxury": 320},
    "amsterdam"     : {"budget": 75,  "mid-range": 150,  "luxury": 380},
    # Middle East
    "dubai"         : {"budget": 80,  "mid-range": 200,  "luxury": 500},
    # Americas
    "new york"      : {"budget": 100, "mid-range": 200,  "luxury": 500},
    # India
    "goa"           : {"budget": 20,  "mid-range": 50,   "luxury": 150},
    "delhi"         : {"budget": 20,  "mid-range": 45,   "luxury": 130},
    "mumbai"        : {"budget": 22,  "mid-range": 50,   "luxury": 140},
    "default"       : {"budget": 50,  "mid-range": 100,  "luxury": 250},
}

@tool
def estimate_budget(query: str) -> str:
    """
    Estimate total travel budget for a trip.
    Use when user asks about travel costs, budget planning, or how much a trip will cost.
    Input format: 'DESTINATION DAYS BUDGET_LEVEL TRAVELERS'
    Example: 'Tokyo 7 mid-range 2' or 'Bali 5 budget 1'
    """
    try:
        parts       = query.strip().split()
        destination = parts[0].lower() if parts else "unknown"
        days        = int(parts[1]) if len(parts) > 1 else 7
        budget      = parts[2].lower() if len(parts) > 2 else "mid-range"
        travelers   = int(parts[3]) if len(parts) > 3 else 1

        # Get cost data for destination
        dest_key = destination
        if dest_key not in DESTINATION_COSTS:
            # Try partial match
            for key in DESTINATION_COSTS:
                if key in destination or destination in key:
                    dest_key = key
                    break
            else:
                dest_key = "default"

        costs = DESTINATION_COSTS.get(dest_key, DESTINATION_COSTS["default"])

        if budget not in costs:
            budget = "mid-range"

        daily_cost      = costs[budget]
        total_per_person = daily_cost * days
        total_all       = total_per_person * travelers

        # Breakdown estimates
        accommodation_pct = 0.35
        food_pct          = 0.25
        activities_pct    = 0.20
        transport_local   = 0.10
        misc_pct          = 0.10

        breakdown = {
            "Accommodation" : total_per_person * accommodation_pct,
            "Food & Dining" : total_per_person * food_pct,
            "Activities"    : total_per_person * activities_pct,
            "Local Transport": total_per_person * transport_local,
            "Misc/Shopping" : total_per_person * misc_pct,
        }

        # Flight estimate (rough)
        flight_estimates = {
            "budget"   : "$200-600 (economy, book early)",
            "mid-range": "$400-1200 (economy/premium economy)",
            "luxury"   : "$1500-5000 (business/first class)"
        }

        report = f"""
💰 BUDGET ESTIMATE: {parts[0].title()} ({days} days)
{'='*45}
👥 Travelers: {travelers} | 💼 Budget Level: {budget.title()}

📊 PER PERSON BREAKDOWN (Daily: ~${daily_cost}/day):
"""
        for category, amount in breakdown.items():
            report += f"   {category:18}: ${amount:,.0f}\n"

        report += f"""
   {'─'*30}
   Total per person  : ${total_per_person:,.0f} (excl. flights)

✈️ FLIGHTS (estimated): {flight_estimates.get(budget, flight_estimates['mid-range'])}

👥 TOTAL FOR {travelers} TRAVELER(S):
   Land costs only   : ${total_all:,.0f}
   With flights (est): ${total_all + (500 * travelers):,.0f} - ${total_all + (1200 * travelers):,.0f}

💡 MONEY-SAVING TIPS:
   • Book accommodation 2-3 months ahead
   • Travel during shoulder season (10-30% cheaper)
   • Use local transport instead of taxis
   • Eat at local restaurants, not tourist spots
   • Get travel insurance ($30-80 for the whole trip)

⚠️  Note: Estimates are approximate. Actual costs vary.
        """

        # Save to database
        save_search("budget", query, f"${total_all:.0f} for {travelers} travelers, {days} days in {parts[0]}")

        return report

    except Exception as e:
        return f"Error estimating budget: {str(e)}"


# Test
print("🧪 Testing Budget Estimator...")
print(estimate_budget.invoke("Tokyo 7 mid-range 2"))

🧪 Testing Budget Estimator...

💰 BUDGET ESTIMATE: Tokyo (7 days)
👥 Travelers: 2 | 💼 Budget Level: Mid-Range

📊 PER PERSON BREAKDOWN (Daily: ~$140/day):
   Accommodation     : $343
   Food & Dining     : $245
   Activities        : $196
   Local Transport   : $98
   Misc/Shopping     : $98

   ──────────────────────────────
   Total per person  : $980 (excl. flights)

✈️ FLIGHTS (estimated): $400-1200 (economy/premium economy)

👥 TOTAL FOR 2 TRAVELER(S):
   Land costs only   : $1,960
   With flights (est): $2,960 - $4,360

💡 MONEY-SAVING TIPS:
   • Book accommodation 2-3 months ahead
   • Travel during shoulder season (10-30% cheaper)
   • Use local transport instead of taxis
   • Eat at local restaurants, not tourist spots
   • Get travel insurance ($30-80 for the whole trip)

⚠️  Note: Estimates are approximate. Actual costs vary.
        


In [ ]:
# ============================================================
# FULL TRAVEL AGENT WITH ALL TOOLS
# ============================================================
# ============================================================
# FULL TRAVEL AGENT WITH ALL TOOLS - FIXED
# ============================================================
import requests
from serpapi import GoogleSearch
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# ── Re-define tools from Week 3-4 ────────────────────────────
@tool
def get_weather(city: str) -> str:
    """Get current weather for a city. Input: city name."""
    CITY_WEATHER = {
        "london"    : {"temp":14, "feel":11, "desc":"Overcast/Light Rain",  "humidity":78, "wind":19},
        "paris"     : {"temp":16, "feel":13, "desc":"Partly Cloudy",        "humidity":72, "wind":15},
        "tokyo"     : {"temp":21, "feel":19, "desc":"Clear Sky",            "humidity":63, "wind":12},
        "bali"      : {"temp":29, "feel":33, "desc":"Sunny with Humidity",  "humidity":82, "wind":10},
        "bangkok"   : {"temp":33, "feel":38, "desc":"Hot & Humid",          "humidity":85, "wind":8},
        "singapore" : {"temp":30, "feel":35, "desc":"Partly Cloudy",        "humidity":84, "wind":11},
        "dubai"     : {"temp":36, "feel":40, "desc":"Sunny & Hot",          "humidity":48, "wind":14},
        "mumbai"    : {"temp":32, "feel":36, "desc":"Hazy & Humid",         "humidity":88, "wind":16},
        "delhi"     : {"temp":28, "feel":30, "desc":"Partly Cloudy",        "humidity":55, "wind":10},
        "new york"  : {"temp":18, "feel":15, "desc":"Mostly Clear",         "humidity":58, "wind":20},
        "goa"       : {"temp":31, "feel":35, "desc":"Sunny & Breezy",       "humidity":75, "wind":18},
        "bangalore" : {"temp":26, "feel":24, "desc":"Pleasant & Cloudy",    "humidity":65, "wind":10},
    }
    city_lower = city.lower().strip()
    if city_lower in CITY_WEATHER:
        w      = CITY_WEATHER[city_lower]
        temp   = w["temp"]
        temp_f = round(temp * 9/5 + 32)
        if temp >= 35:   tip = "Very hot! Carry water and sunscreen."
        elif temp >= 28: tip = "Warm! Light clothes recommended."
        elif temp >= 20: tip = "Great weather for sightseeing!"
        elif temp >= 12: tip = "Mild, carry a light jacket."
        else:            tip = "Cold! Pack warm clothes."
        return (
            "Weather in " + city.title() + ":\n" +
            "Temperature: " + str(temp) + "C / " + str(temp_f) + "F\n" +
            "Condition: " + w["desc"] + "\n" +
            "Humidity: " + str(w["humidity"]) + "%\n" +
            "Travel Tip: " + tip
        )
    try:
        resp = llm.invoke("Give weather for " + city.title() + " briefly: temp, condition, humidity, one tip.")
        return resp.content.strip()
    except:
        return "Weather unavailable for " + city.title()

@tool
def convert_currency(query: str) -> str:
    """Convert currency. Input: 'amount FROM TO' e.g. '100 USD INR'"""
    try:
        p              = query.strip().split()
        amount, fr, to = float(p[0]), p[1].upper(), p[2].upper()
        r = requests.get(
            f"https://v6.exchangerate-api.com/v6/{EXCHANGERATE_API_KEY}/pair/{fr}/{to}/{amount}",
            timeout=10
        ).json()
        if r.get("result") == "success":
            return f"{amount:,.2f} {fr} = {r['conversion_result']:,.2f} {to} (Rate: {r['conversion_rate']:.4f})"
        return f"Could not convert {fr} to {to}."
    except Exception as e:
        return f"Currency error: {str(e)[:80]}"

@tool
def search_travel_info(query: str) -> str:
    """Search the web for any travel information. Input: travel topic or question."""
    try:
        results = GoogleSearch({
            "engine": "google", "q": f"travel {query}",
            "api_key": SERP_API_KEY, "num": 4
        }).get_dict()
        output = f"Search: {query}\n"
        if "answer_box" in results:
            ab = results["answer_box"]
            output += f"Quick Answer: {ab.get('answer', ab.get('snippet',''))[:300]}\n"
        for r in results.get("organic_results", [])[:3]:
            output += f"- {r.get('title','')[:70]}\n  {r.get('snippet','')[:150]}\n"
        return output
    except Exception as e:
        return f"Search error: {str(e)[:80]}"

# ── All tools combined ────────────────────────────────────────
all_tools  = [
    get_weather,
    convert_currency,
    search_travel_info,
    generate_itinerary,   # from previous cell
    estimate_budget,      # from previous cell
]
TOOL_MAP       = {tool.name: tool for tool in all_tools}
llm_with_tools = llm.bind_tools(all_tools)

print("✅ All tools ready:", list(TOOL_MAP.keys()))

# ── Agent function ────────────────────────────────────────────
def ask_agent(question: str) -> str:
    """Smart travel agent using tool binding."""
    print(f"Processing: {question[:60]}...")

    messages = [
        SystemMessage(content=(
            "You are TravelBot, an expert AI Travel Concierge. "
            "Use tools to get real information. "
            "IMPORTANT: Call only ONE tool at a time. "
            "Be friendly, practical and helpful."
        )),
        HumanMessage(content=question)
    ]

    try:
        response = llm_with_tools.invoke(messages)

        if not hasattr(response, "tool_calls") or not response.tool_calls:
            return response.content

        messages.append(response)
        all_results = []

        for tool_call in response.tool_calls:
            name    = tool_call.get("name", "")
            args    = tool_call.get("args", {})
            call_id = tool_call.get("id", "tool_1")

            print(f"  Using: {name}")
            input_val = list(args.values())[0] if isinstance(args, dict) and args else str(args)

            try:
                result = TOOL_MAP[name].invoke(str(input_val)) \
                         if name in TOOL_MAP else f"Tool '{name}' not found"
            except Exception as e:
                result = f"Tool error: {str(e)[:100]}"

            all_results.append(f"[{name}]:\n{result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call_id))

        # Final answer using tool results
        combined     = "\n\n".join(all_results)
        final_prompt = (
            f"You are TravelBot. Answer: '{question}'\n\n"
            f"Data from tools:\n{combined}\n\n"
            f"Use ALL the data. Be specific with numbers/prices. "
            f"Add 2-3 practical tips. Be friendly and complete."
        )
        return llm.invoke(final_prompt).content

    except Exception as e:
        error = str(e)
        if "tool_use_failed" in error or "Failed to call" in error:
            print("Tool failed, using direct LLM...")
            return llm.invoke(
                f"As travel expert answer: {question}\nBe specific with tips and costs."
            ).content
        return f"Error: {error[:150]}"


# ── Wrapper class ─────────────────────────────────────────────
class TravelAgentExecutor:
    def invoke(self, inputs: dict) -> dict:
        return {"output": ask_agent(inputs.get("input", ""))}

agent_executor = TravelAgentExecutor()
print("✅ Full Travel Agent ready with", len(all_tools), "tools!")

# ── Test ──────────────────────────────────────────────────────
print("\nTesting agent...\n" + "="*50)
result = agent_executor.invoke({
    "input": "Plan a 3-day trip to Bali for 2 people with budget"
})
print(result["output"])

✅ All tools ready: ['get_weather', 'convert_currency', 'search_travel_info', 'generate_itinerary', 'estimate_budget']
✅ Full Travel Agent ready with 5 tools!

Testing agent...
Processing: Plan a 3-day trip to Bali for 2 people with budget budget...
  Using: generate_itinerary
Hello and welcome to TravelBot. I'm excited to help you plan a 3-day trip to Bali for 2 people on a budget. Based on our research, I've put together an itinerary that should fit your needs and provide an unforgettable experience.

Our 3-day itinerary includes a mix of relaxation, culture, and adventure, and we've estimated the costs to help you plan your trip. Here's a breakdown of what you can expect:

**Day 1: South Kuta Beach**
- Morning: Start the day with breakfast at Naughty Nuri's, where you can enjoy a delicious meal for $5-$7 per person. Try their traditional Balinese coffee and a plate of nasi goreng. Be sure to arrive early to avoid the crowds.
- Afternoon: Spend the day relaxing on Kuta Beach, where yo

In [ ]:
# ============================================================
# COMPLETE TRAVEL PLANNER GRADIO UI
# ============================================================
import gradio as gr

def agent_respond(message, history):
    """Main chat function."""
    if not message.strip():
        return "", history
    try:
        result   = agent_executor.invoke({"input": message})
        response = result["output"]
        save_search("chat", message[:100], response[:200])
    except Exception as e:
        response = f"Sorry, I ran into an issue: {str(e)[:200]}\n\nPlease try rephrasing your question."
    history.append((message, response))
    return "", history


def show_history():
    """Show saved search history from database."""
    rows = get_search_history(15)
    if not rows:
        return "No searches saved yet. Start chatting!"
    result = "📋 Recent Search History:\n" + "="*40 + "\n"
    for row in rows:
        result += f"[{row[2]}] {row[0].upper()}: {row[1]}\n"
    return result


def show_itineraries():
    """Show saved itineraries from database."""
    rows = get_saved_itineraries()
    if not rows:
        return "No itineraries saved yet. Ask me to plan a trip!"
    result = "🗺️ Saved Itineraries:\n" + "="*40 + "\n"
    for row in rows:
        result += f"• {row[0]} — {row[1]} days, {row[2]}, {row[3]} travelers | Saved: {row[4]}\n"
    return result


def quick_itinerary(destination, days, budget, travelers):
    """Generate itinerary from the quick planner form."""
    if not destination:
        return "Please enter a destination!"
    query    = f"{destination} {days} {budget} {travelers}"
    result   = generate_itinerary.invoke(query)
    return result


def quick_budget(destination, days, budget, travelers):
    """Estimate budget from the quick planner form."""
    if not destination:
        return "Please enter a destination!"
    query  = f"{destination} {days} {budget} {travelers}"
    result = estimate_budget.invoke(query)
    return result


# ── Build the Gradio App ─────────────────────────────────────
with gr.Blocks(
    title="AI Travel Concierge",
    theme=gr.themes.Soft(primary_hue="blue"),
    css=".container { max-width: 1200px; }"
) as app:

    gr.Markdown("""
    # ✈️ AI Travel Concierge
    ### Powered by Google Gemini | Plan your perfect trip with AI
    🌍 Weather &nbsp; ✈️ Flights &nbsp; 🏨 Hotels &nbsp; 🗓️ Itineraries &nbsp; 💰 Budgets &nbsp; 💱 Currency
    """)

    with gr.Tabs():

        # ── TAB 1: Chat ─────────────────────────────────────
        with gr.Tab("💬 Chat with Concierge"):
            gr.Markdown("### Ask me anything about travel!")

            chatbot = gr.Chatbot(
                value=[(None, "👋 Hello! I'm your AI Travel Concierge!\n\nI can help you:\n✈️ Plan complete trips with day-by-day itineraries\n💰 Estimate travel budgets\n🌤️ Check weather at destinations\n💱 Convert currencies\n🏨 Find hotels & flights\n\nTry: *'Plan a 5-day trip to Bali for 2 people with mid-range budget'*")],
                height=480,
                label="Travel Assistant"
            )

            with gr.Row():
                chat_input = gr.Textbox(
                    placeholder="e.g. Plan a 7-day Japan trip for 2 people, mid-range budget",
                    label="Your question",
                    scale=5
                )
                chat_btn = gr.Button("Send ✈️", variant="primary", scale=1)

            gr.Markdown("**💡 Try these:**")
            with gr.Row():
                samples = [
                    "Plan 5-day Bali trip for 2, budget",
                    "How much will Tokyo cost for 7 days?",
                    "Weather in Paris right now?",
                    "Convert 1000 USD to Thai Baht",
                    "Best time to visit Japan?",
                    "Visa requirements for Dubai from India"
                ]
                for s in samples:
                    gr.Button(s, size="sm").click(lambda x=s: x, outputs=chat_input)

            chat_btn.click(agent_respond, [chat_input, chatbot], [chat_input, chatbot])
            chat_input.submit(agent_respond, [chat_input, chatbot], [chat_input, chatbot])

        # ── TAB 2: Trip Planner ─────────────────────────────
        with gr.Tab("🗓️ Trip Planner"):
            gr.Markdown("### Quick Trip Planner — Fill in the details!")

            with gr.Row():
                with gr.Column():
                    dest_input     = gr.Textbox(label="🌍 Destination", placeholder="e.g. Tokyo, Bali, Paris")
                    days_input     = gr.Slider(1, 14, value=5, step=1, label="📅 Number of Days")
                    budget_input   = gr.Radio(["budget", "mid-range", "luxury"], value="mid-range", label="💰 Budget Level")
                    travelers_in   = gr.Slider(1, 10, value=2, step=1, label="👥 Number of Travelers")

                    with gr.Row():
                        itin_btn   = gr.Button("📍 Generate Itinerary", variant="primary")
                        budget_btn = gr.Button("💰 Estimate Budget", variant="secondary")

                with gr.Column():
                    plan_output = gr.Textbox(
                        label="📋 Result",
                        lines=25,
                        placeholder="Your itinerary or budget estimate will appear here..."
                    )

            itin_btn.click(
                quick_itinerary,
                [dest_input, days_input, budget_input, travelers_in],
                plan_output
            )
            budget_btn.click(
                quick_budget,
                [dest_input, days_input, budget_input, travelers_in],
                plan_output
            )

        # ── TAB 3: Saved Data ──────────────────────────────
        with gr.Tab("💾 Saved Data"):
            gr.Markdown("### Your saved searches and itineraries from this session")

            with gr.Row():
                history_btn    = gr.Button("📋 Show Search History", variant="secondary")
                itinerary_btn  = gr.Button("🗺️ Show Saved Itineraries", variant="secondary")

            saved_output = gr.Textbox(
                label="Saved Data",
                lines=20,
                placeholder="Click a button to view saved data..."
            )

            history_btn.click(show_history, outputs=saved_output)
            itinerary_btn.click(show_itineraries, outputs=saved_output)

        # ── TAB 4: About ───────────────────────────────────
        with gr.Tab("ℹ️ About"):
            gr.Markdown("""
            ## AI Travel Concierge - Project Info

            ### 🏗️ Architecture
            - **LLM**: Google Gemini 1.5 Flash
            - **Framework**: LangChain (ReAct Agent)
            - **Database**: SQLite (persistent storage)
            - **UI**: Gradio (runs in Google Colab)

            ### 🛠️ Tools
            | Tool | API | Purpose |
            |------|-----|--------|
            | Weather | Weatherstack | Real-time weather |
            | Currency | ExchangeRate-API | Currency conversion |
            | Flights | SERP API | Flight search |
            | Hotels | SERP API | Hotel search |
            | Web Search | SERP API | General travel info |
            | Itinerary | Gemini LLM | Day-wise trip planning |
            | Budget | Built-in DB | Cost estimation |

            ### 📅 Development Timeline
            - **Week 1-2**: Foundation & RAG chatbot
            - **Week 3-4**: Agent + API tools
            - **Week 5-6**: Specialization + Database
            - **Week 7-8**: Polish & Final deployment

            ### 👨‍💻 Track A Project | Internship 2024
            """)

app.launch(share=True, debug=False)
print("\n🔗 Complete Travel Concierge is LIVE!")

/tmp/ipykernel_2711/914187760.py:61: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_2711/914187760.py:61: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_2711/914187760.py:79: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_2711/914187760.py:79: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you wan

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cfd62cfa642e2a2626.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🔗 Complete Travel Concierge is LIVE!


**WEEK 7-8 TASK**

In [1]:
!pip install -q --upgrade langchain langchain-core
!pip install -q langchain-groq langchain-community
!pip install -q google-search-results requests
print("✅ Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 12.8 MB/s eta 0:00:00
✅ Done!


In [2]:
!pip install -q langchain langchain-google-genai langchain-community
!pip install -q google-generativeai google-search-results gradio requests faiss-cpu
!pip install -q langchain langchain-core langchain-groq
!pip install -q langchain-community google-search-results requests
print("✅ Done!")
print("✅ Dependencies ready!")

✅ Done!
✅ Dependencies ready!


In [4]:
# ============================================================
# SETUP - Run after fresh restart
# ============================================================
import os, sqlite3, requests, json, time
from datetime import datetime

# ── API Keys ──────────────────────────────────────────────────
try:
    from google.colab import userdata
    GROQ_API_KEY         = userdata.get('GROQ_API_KEY')
    SERP_API_KEY         = userdata.get('SERP_API_KEY')
    WEATHERSTACK_API_KEY = userdata.get('WEATHERSTACK_API_KEY')
    EXCHANGERATE_API_KEY = userdata.get('EXCHANGERATE_API_KEY')
    print("✅ API Keys loaded!")
except:
    GROQ_API_KEY         = "GROQ_API_KEY"
    SERP_API_KEY         = "SERP_API_KEY"
    WEATHERSTACK_API_KEY = "WEATHERSTACK_API_KEY"
    EXCHANGERATE_API_KEY = "EXCHANGERATE_API_KEY"

os.environ["SERP_API_KEY"] = SERP_API_KEY

# ── LLM (import AFTER fresh install) ─────────────────────────
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from serpapi import GoogleSearch

GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "llama-3.1-70b-versatile",
    "llama-3.1-8b-instant",
    "mixtral-8x7b-32768",
    "gemma2-9b-it",
]

llm = None
for model in GROQ_MODELS:
    try:
        test = ChatGroq(model=model, temperature=0.7, groq_api_key=GROQ_API_KEY)
        test.invoke("Hi")
        llm = test
        print(f"✅ LLM ready: {model}")
        break
    except Exception as e:
        print(f"❌ {model}: {str(e)[:50]}")

# ── Database ──────────────────────────────────────────────────
DB_PATH = "/content/travel_concierge.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    c    = conn.cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS search_history (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        search_type TEXT, query TEXT, result TEXT,
        timestamp TEXT DEFAULT (datetime('now')))""")
    c.execute("""CREATE TABLE IF NOT EXISTS itineraries (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        destination TEXT, duration_days INTEGER,
        budget_level TEXT, travelers INTEGER,
        itinerary_text TEXT, estimated_cost TEXT,
        created_at TEXT DEFAULT (datetime('now')))""")
    conn.commit()
    conn.close()

def save_search(t, q, r):
    try:
        conn = sqlite3.connect(DB_PATH)
        conn.execute("INSERT INTO search_history (search_type, query, result) VALUES (?,?,?)", (t, q, r[:500]))
        conn.commit(); conn.close()
    except: pass

def save_itinerary(dest, days, budget, travelers, text, cost):
    try:
        conn = sqlite3.connect(DB_PATH)
        conn.execute(
            "INSERT INTO itineraries (destination,duration_days,budget_level,travelers,itinerary_text,estimated_cost) VALUES (?,?,?,?,?,?)",
            (dest, days, budget, travelers, text, cost)
        )
        conn.commit(); conn.close()
    except: pass

def get_search_history(limit=10):
    conn  = sqlite3.connect(DB_PATH)
    rows  = conn.execute("SELECT search_type, query, timestamp FROM search_history ORDER BY id DESC LIMIT ?", (limit,)).fetchall()
    conn.close()
    return rows

def get_saved_itineraries():
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("SELECT destination, duration_days, budget_level, travelers, created_at FROM itineraries ORDER BY id DESC").fetchall()
    conn.close()
    return rows

init_db()
print("✅ Database ready!")
print(f"✅ Setup complete! LLM: {llm.model_name if llm else 'FAILED'}")

✅ API Keys loaded!
✅ LLM ready: llama-3.3-70b-versatile
✅ Database ready!
✅ Setup complete! LLM: llama-3.3-70b-versatile


In [8]:
# ============================================================
# ROBUST TOOL DEFINITIONS WITH FULL ERROR HANDLING
# ============================================================

DESTINATION_COSTS = {
    "bali":{"budget":30,"mid-range":70,"luxury":200},
    "bangkok":{"budget":25,"mid-range":60,"luxury":180},
    "tokyo":{"budget":70,"mid-range":140,"luxury":350},
    "japan":{"budget":70,"mid-range":140,"luxury":350},
    "paris":{"budget":80,"mid-range":160,"luxury":400},
    "london":{"budget":90,"mid-range":180,"luxury":450},
    "dubai":{"budget":80,"mid-range":200,"luxury":500},
    "singapore":{"budget":80,"mid-range":150,"luxury":350},
    "goa":{"budget":20,"mid-range":50,"luxury":150},
    "default":{"budget":50,"mid-range":100,"luxury":250}
}

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city. Input: city name."""
    try:
        r = requests.get(
            "http://api.weatherstack.com/current",
            params={"access_key": WEATHERSTACK_API_KEY, "query": city, "units": "m"},
            timeout=10
        )
        r.raise_for_status()
        d = r.json()
        if "current" not in d:
            return f"Weather unavailable for {city}. API response: {str(d)[:100]}"
        c = d["current"]; l = d.get("location", {})
        return (f"🌍 {l.get('name',city)}, {l.get('country','')}: "
                f"{c.get('temperature')}°C, {', '.join(c.get('weather_descriptions',['N/A']))}, "
                f"Humidity: {c.get('humidity')}%, Wind: {c.get('wind_speed')}km/h")
    except requests.Timeout:
        return f"Weather service timed out for {city}. Try again."
    except requests.HTTPError as e:
        return f"Weather API error (HTTP {e.response.status_code}) for {city}."
    except Exception as e:
        return f"Weather error for {city}: {str(e)[:80]}"

@tool
def convert_currency(query: str) -> str:
    """Convert currency. Input: 'amount FROM TO' e.g. '100 USD INR'"""
    try:
        p = query.strip().split()
        if len(p) < 3:
            return "Format: 'amount FROM TO' e.g. '100 USD INR'"
        amount, fr, to = float(p[0]), p[1].upper(), p[2].upper()
        r = requests.get(
            f"https://v6.exchangerate-api.com/v6/{EXCHANGERATE_API_KEY}/pair/{fr}/{to}/{amount}",
            timeout=10
        )
        d = r.json()
        if d.get("result") == "success":
            return f"💱 {amount:,.2f} {fr} = {d['conversion_result']:,.2f} {to} (Rate: {d['conversion_rate']:.4f})"
        fallback = {("USD","INR"):83.5,("EUR","INR"):90.2,("USD","JPY"):149.5,("USD","THB"):35.2}
        rate = fallback.get((fr, to))
        if rate:
            return f"💱 ~{amount*rate:,.2f} {to} (approx, offline rate)"
        return f"Cannot convert {fr}→{to}: {d.get('error-type','unknown error')}"
    except ValueError:
        return "Invalid amount. Use: '100 USD INR'"
    except Exception as e:
        return f"Currency error: {str(e)[:80]}"

@tool
def search_travel_info(query: str) -> str:
    """Search the web for travel information. Input: any travel topic."""
    try:
        results = GoogleSearch({"engine":"google","q":f"travel {query} guide","api_key":SERP_API_KEY,"num":4}).get_dict()
        out = f"🔍 {query}:\n"
        if "answer_box" in results:
            ab = results["answer_box"]
            out += f"📌 {ab.get('answer', ab.get('snippet',''))[:300]}\n\n"
        for r in results.get("organic_results",[])[:3]:
            out += f"• {r.get('title','')}\n  {r.get('snippet','')[:200]}\n\n"
        return out or f"No results found for: {query}"
    except Exception as e:
        return f"Search error: {str(e)[:80]}"

@tool
def generate_itinerary(query: str) -> str:
    """Generate a day-by-day trip itinerary. Format: 'DESTINATION DAYS BUDGET TRAVELERS' e.g. 'Tokyo 5 mid-range 2'"""
    try:
        p = query.strip().split()
        dest, days, budget, travelers = p[0], int(p[1]) if len(p)>1 else 5, p[2] if len(p)>2 else "mid-range", int(p[3]) if len(p)>3 else 1
        days = min(max(days, 1), 14)
        prompt = f"""Create a practical {days}-day itinerary for {dest}. Budget: {budget}. Travelers: {travelers}.
For each day include: Morning/Afternoon/Evening activities, food recommendations, transport tips, estimated daily cost in USD.
Format clearly with Day 1, Day 2, etc. Include 1-2 pro tips per day. Be specific and practical."""
        response = llm.invoke(prompt)
        itinerary = f"\n🗺️ {days}-DAY {dest.upper()} ITINERARY\n👥 {travelers} Travelers | 💰 {budget.title()} Budget\n{'='*50}\n" + response.content
        save_itinerary(dest, days, budget, travelers, response.content, f"{budget} for {travelers}")
        save_search("itinerary", query, f"{days}-day {dest} itinerary created")
        return itinerary
    except Exception as e:
        return f"Itinerary error: {str(e)[:80]}"

@tool
def estimate_budget(query: str) -> str:
    """Estimate travel budget. Format: 'DESTINATION DAYS BUDGET TRAVELERS' e.g. 'Bali 7 budget 2'"""
    try:
        p = query.strip().split()
        dest, days, budget, travelers = p[0].lower(), int(p[1]) if len(p)>1 else 7, p[2].lower() if len(p)>2 else "mid-range", int(p[3]) if len(p)>3 else 1
        dk = dest if dest in DESTINATION_COSTS else next((k for k in DESTINATION_COSTS if k in dest or dest in k), "default")
        costs = DESTINATION_COSTS.get(dk, DESTINATION_COSTS["default"])
        if budget not in costs: budget = "mid-range"
        daily = costs[budget]; total_pp = daily * days; total = total_pp * travelers
        save_search("budget", query, f"${total:.0f} total for {travelers} travelers")
        return f"""
💰 Budget: {p[0].title()} | {days} days | {budget.title()} | {travelers} traveler(s)
{'='*45}
Daily cost/person : ~${daily}/day
Land costs/person : ${total_pp:,.0f}
Total ({travelers} people)  : ${total:,.0f} (excl. flights)
With flights (est) : ${total + 600*travelers:,.0f} – ${total + 1400*travelers:,.0f}

Breakdown/person:
  🏨 Accommodation : ${total_pp*0.35:,.0f}
  🍜 Food          : ${total_pp*0.25:,.0f}
  🎭 Activities    : ${total_pp*0.20:,.0f}
  🚌 Local travel  : ${total_pp*0.10:,.0f}
  🛍️ Misc          : ${total_pp*0.10:,.0f}

💡 Tips: Travel shoulder season, book ahead, eat local!
        """
    except Exception as e:
        return f"Budget error: {str(e)[:80]}"



ALL_TOOLS  = [get_weather, convert_currency, search_travel_info, generate_itinerary, estimate_budget]
TOOL_MAP   = {tool.name: tool for tool in ALL_TOOLS}

# ✅ Use bind_tools instead of create_react_agent
llm_with_tools = llm.bind_tools(ALL_TOOLS)

# ── Agent function ────────────────────────────────────────────
def ask_agent(question: str) -> str:
    """Smart travel agent using tool binding."""
    print(f"Processing: {question[:60]}...")

    messages = [
        SystemMessage(content=(
            "You are TravelBot, an expert AI Travel Concierge. "
            "Use tools to get real information. "
            "IMPORTANT: Call only ONE tool at a time. "
            "Be friendly, practical and helpful."
        )),
        HumanMessage(content=question)
    ]

    try:
        response = llm_with_tools.invoke(messages)

        if not hasattr(response, "tool_calls") or not response.tool_calls:
            return response.content

        messages.append(response)
        all_results = []

        for tool_call in response.tool_calls:
            name    = tool_call.get("name", "")
            args    = tool_call.get("args", {})
            call_id = tool_call.get("id", "tool_1")

            print(f"  Using: {name}")
            input_val = list(args.values())[0] if isinstance(args, dict) and args else str(args)

            try:
                result = TOOL_MAP[name].invoke(str(input_val)) \
                         if name in TOOL_MAP else f"Tool '{name}' not found"
            except Exception as e:
                result = f"Tool error: {str(e)[:100]}"

            all_results.append(f"[{name}]:\n{result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call_id))

        # Final answer using tool results
        combined     = "\n\n".join(all_results)
        final_prompt = (
            f"You are TravelBot. Answer: '{question}'\n\n"
            f"Data from tools:\n{combined}\n\n"
            f"Use ALL the data. Be specific with numbers/prices. "
            f"Add 2-3 practical tips. Be friendly and complete."
        )
        return llm.invoke(final_prompt).content

    except Exception as e:
        error = str(e)
        if "tool_use_failed" in error or "Failed to call" in error:
            print("Tool failed, using direct LLM...")
            return llm.invoke(
                f"As travel expert answer: {question}\nBe specific with tips and costs."
            ).content
        return f"Error: {error[:150]}"


# ── Wrapper class ─────────────────────────────────────────────
class TravelAgentExecutor:
    def invoke(self, inputs: dict) -> dict:
        return {"output": ask_agent(inputs.get("input", ""))}

agent_executor = TravelAgentExecutor()
print("✅ Full agent ready with", len(ALL_TOOLS), "tools!")
print("🛠️  Tools:", list(TOOL_MAP.keys()))

# ── Quick test ────────────────────────────────────────────────
print("\nTesting...")
result = agent_executor.invoke({"input": "What is the budget for 5 days in Thailand for 2 people?"})
print(result["output"])

✅ Full agent ready with 5 tools!
🛠️  Tools: ['get_weather', 'convert_currency', 'search_travel_info', 'generate_itinerary', 'estimate_budget']

Testing...
Processing: What is the budget for 5 days in Thailand for 2 people?...
  Using: estimate_budget
Thailand, what a fantastic destination.  I'd be happy to help you estimate your budget for a 5-day trip to Thailand for 2 people.

According to my calculations, your daily cost per person would be around $100 per day. For 5 days, the land costs per person would be approximately $500. So, for 2 people, the total land cost would be $1,000 (excluding flights).

If you factor in flights, your total estimated cost for the trip would be between $2,200 and $3,800.

Here's a breakdown of the estimated costs per person:
- Accommodation: $175
- Food: $125
- Activities: $100
- Local travel: $50
- Miscellaneous: $50

To make the most of your trip and stay within your budget, here are some practical tips:
1. **Travel during the shoulder season**: Price

In [9]:
# ============================================================
# AUTOMATED TEST SUITE
# ============================================================
import time

test_cases = [
    # (test_name, question, expected_keywords)
    ("Weather Test",      "What's the weather in Tokyo?",                ["temperature", "°C", "humidity"]),
    ("Currency Test",     "Convert 500 USD to Japanese Yen",             ["JPY", "USD", "Rate"]),
    ("Budget Test",       "How much does a 5-day Bali trip cost?",        ["$", "Budget", "days"]),
    ("Itinerary Test",    "Plan a 3-day Paris trip for 1 person",         ["Day 1", "Day 2", "Day 3"]),
    ("Search Test",       "Visa requirements for Japan from India",       ["visa", "Japan", "India"]),
]

print("🧪 Running Automated Tests...")
print("=" * 60)

results = []
for test_name, question, expected_kw in test_cases:
    print(f"\n⏳ {test_name}: '{question[:50]}...'")
    start = time.time()
    try:
        result   = agent_executor.invoke({"input": question})
        answer   = result["output"]
        elapsed  = time.time() - start

        # Check if expected keywords appear in response (case-insensitive)
        answer_lower = answer.lower()
        kw_found     = [kw for kw in expected_kw if kw.lower() in answer_lower]
        passed       = len(kw_found) >= len(expected_kw) // 2  # Pass if ≥ half keywords found

        status = "✅ PASS" if passed else "⚠️ PARTIAL"
        print(f"{status} | {elapsed:.1f}s | Keywords found: {kw_found}")
        results.append((test_name, passed, elapsed, answer[:100]))

    except Exception as e:
        elapsed = time.time() - start
        print(f"❌ FAIL | {elapsed:.1f}s | Error: {str(e)[:80]}")
        results.append((test_name, False, elapsed, str(e)))

# Summary
print("\n" + "="*60)
print("📊 TEST SUMMARY")
print("="*60)
passed_count = sum(1 for _, p, _, _ in results if p)
for name, passed, elapsed, _ in results:
    status = "✅" if passed else "❌"
    print(f"{status} {name:20} {elapsed:.1f}s")

print(f"\n🎯 Score: {passed_count}/{len(results)} tests passed")
avg_time = sum(t for _,_,t,_ in results) / len(results)
print(f"⏱️  Average response time: {avg_time:.1f}s")

🧪 Running Automated Tests...

⏳ Weather Test: 'What's the weather in Tokyo?...'
Processing: What's the weather in Tokyo?...
  Using: get_weather
✅ PASS | 2.0s | Keywords found: ['temperature', '°C', 'humidity']

⏳ Currency Test: 'Convert 500 USD to Japanese Yen...'
Processing: Convert 500 USD to Japanese Yen...
  Using: convert_currency
✅ PASS | 1.9s | Keywords found: ['JPY', 'USD', 'Rate']

⏳ Budget Test: 'How much does a 5-day Bali trip cost?...'
Processing: How much does a 5-day Bali trip cost?...
  Using: estimate_budget
✅ PASS | 1.7s | Keywords found: ['$', 'Budget']

⏳ Itinerary Test: 'Plan a 3-day Paris trip for 1 person...'
Processing: Plan a 3-day Paris trip for 1 person...
  Using: generate_itinerary
✅ PASS | 5.0s | Keywords found: ['Day 1', 'Day 2', 'Day 3']

⏳ Search Test: 'Visa requirements for Japan from India...'
Processing: Visa requirements for Japan from India...
  Using: search_travel_info
✅ PASS | 3.8s | Keywords found: ['visa', 'Japan', 'India']

📊 TEST SUMMARY
✅ W

In [10]:
# ============================================================
# EXPORT ITINERARY TO TEXT FILE
# ============================================================

def export_itinerary_to_file(destination: str, days: int, budget: str, travelers: int) -> str:
    """Generate and save itinerary to a downloadable .txt file."""
    query    = f"{destination} {days} {budget} {travelers}"
    itinerary = generate_itinerary.invoke(query)

    # Create file
    filename = f"/content/{destination.replace(' ','_')}_{days}day_itinerary.txt"
    with open(filename, "w") as f:
        f.write(f"AI TRAVEL CONCIERGE - TRIP ITINERARY\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
        f.write("=" * 60 + "\n\n")
        f.write(itinerary)
        f.write("\n\n" + "=" * 60 + "\n")
        f.write("Generated by AI Travel Concierge | Powered by Google Gemini\n")

    # Also get budget estimate
    budget_est = estimate_budget.invoke(query)
    with open(filename, "a") as f:
        f.write("\nBUDGET ESTIMATE\n" + "="*60 + "\n")
        f.write(budget_est)

    print(f"✅ Itinerary saved to: {filename}")
    print("📥 In Colab: Files panel (left sidebar) → download the .txt file")
    return filename


# Demo export
export_itinerary_to_file("Thailand", 7, "mid-range", 2)

✅ Itinerary saved to: /content/Thailand_7day_itinerary.txt
📥 In Colab: Files panel (left sidebar) → download the .txt file


'/content/Thailand_7day_itinerary.txt'

In [11]:
# ============================================================
# FINAL PRODUCTION-READY APPLICATION
# ============================================================
import gradio as gr

def agent_chat(message, history):
    if not message.strip():
        return "", history
    try:
        result   = agent_executor.invoke({"input": message})
        response = result["output"]
        save_search("chat", message[:100], response[:200])
    except Exception as e:
        response = f"⚠️ I encountered an issue: {str(e)[:150]}\n\nPlease try rephrasing your question."
    history.append((message, response))
    return "", history

def quick_plan(dest, days, budget, travelers):
    if not dest.strip():
        return "Please enter a destination.", ""
    q        = f"{dest} {int(days)} {budget} {int(travelers)}"
    itin     = generate_itinerary.invoke(q)
    bud      = estimate_budget.invoke(q)
    return itin, bud

def export_plan(dest, days, budget, travelers):
    if not dest.strip():
        return "Please enter a destination first."
    path = export_itinerary_to_file(dest, int(days), budget, int(travelers))
    return f"✅ Saved to: {path}\n📥 Download from Colab Files panel (left sidebar)"

def view_db_history():
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("SELECT search_type, query, timestamp FROM search_history ORDER BY id DESC LIMIT 20").fetchall()
    conn.close()
    if not rows: return "No history yet."
    return "\n".join([f"[{r[2][:16]}] {r[0].upper()}: {r[1][:60]}" for r in rows])

def view_saved_itineraries():
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("SELECT destination, duration_days, budget_level, travelers, created_at FROM itineraries ORDER BY id DESC").fetchall()
    conn.close()
    if not rows: return "No itineraries saved yet."
    return "\n".join([f"📍 {r[0]} | {r[1]} days | {r[2]} | {r[3]} travelers | {r[4][:16]}" for r in rows])


# ── FINAL GRADIO APP ──────────────────────────────────────────
css = """
.title-block { text-align: center; padding: 20px; }
.tab-nav { font-size: 16px; }
"""

with gr.Blocks(title="AI Travel Concierge", theme=gr.themes.Soft(primary_hue="blue"), css=css) as final_app:

    with gr.Row(elem_classes="title-block"):
        gr.Markdown("""
        # ✈️ AI Travel Concierge
        **Powered by Google Gemini · LangChain · Real APIs**
        🌍 Weather &nbsp;|&nbsp; ✈️ Flights &nbsp;|&nbsp; 🏨 Hotels &nbsp;|&nbsp; 🗓️ Itineraries &nbsp;|&nbsp; 💰 Budgets &nbsp;|&nbsp; 💱 Currency
        """)

    with gr.Tabs(elem_classes="tab-nav"):

        # TAB 1: CHAT
        with gr.Tab("💬 Chat"):
            bot = gr.Chatbot(
                value=[(None, """👋 **Welcome to AI Travel Concierge!**

I can help you with:
✈️ **Flights** — search routes and booking tips
🏨 **Hotels** — find accommodation by budget
🗓️ **Itineraries** — day-by-day trip plans
💰 **Budgets** — estimate total trip costs
🌤️ **Weather** — real-time conditions
💱 **Currency** — live exchange rates

Try: *"Plan a 5-day Bali trip for 2 people on a mid-range budget"*""")],
                height=500, label=""
            )
            with gr.Row():
                msg = gr.Textbox(placeholder="Ask anything about travel...", scale=5, label="", show_label=False)
                btn = gr.Button("Send ✈️", variant="primary", scale=1)
            gr.Markdown("**💡 Quick prompts:**")
            with gr.Row():
                for q in ["Plan 5-day Bali trip, 2 people budget", "Weather in Tokyo now", "Convert 500 USD to Euro", "Visa for Japan from India"]:
                    gr.Button(q, size="sm").click(lambda x=q: x, outputs=msg)
            btn.click(agent_chat, [msg, bot], [msg, bot])
            msg.submit(agent_chat, [msg, bot], [msg, bot])

        # TAB 2: TRIP PLANNER
        with gr.Tab("🗓️ Trip Planner"):
            gr.Markdown("### Fill in details → Get your complete trip plan instantly!")
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("#### Trip Details")
                    d_dest     = gr.Textbox(label="🌍 Destination", placeholder="e.g. Bali, Tokyo, Paris")
                    d_days     = gr.Slider(1, 14, value=5, step=1, label="📅 Days")
                    d_budget   = gr.Radio(["budget", "mid-range", "luxury"], value="mid-range", label="💰 Budget")
                    d_travs    = gr.Slider(1, 10, value=2, step=1, label="👥 Travelers")
                    with gr.Row():
                        plan_btn   = gr.Button("🗓️ Plan My Trip!", variant="primary")
                        export_btn = gr.Button("💾 Save to File", variant="secondary")
                    export_out = gr.Textbox(label="Export Status", lines=2)

                with gr.Column(scale=2):
                    with gr.Tabs():
                        with gr.Tab("📍 Itinerary"):
                            itin_out = gr.Textbox(label="", lines=28, show_label=False)
                        with gr.Tab("💰 Budget"):
                            bud_out  = gr.Textbox(label="", lines=28, show_label=False)

            plan_btn.click(quick_plan, [d_dest,d_days,d_budget,d_travs], [itin_out, bud_out])
            export_btn.click(export_plan, [d_dest,d_days,d_budget,d_travs], export_out)

        # TAB 3: HISTORY
        with gr.Tab("💾 History"):
            gr.Markdown("### Searches and itineraries saved this session")
            with gr.Row():
                gr.Button("📋 Search History",   variant="secondary").click(view_db_history,        outputs=gr.Textbox(lines=15, label="History"))
                gr.Button("🗺️ Saved Itineraries", variant="secondary").click(view_saved_itineraries, outputs=gr.Textbox(lines=15, label="Itineraries"))

        # TAB 4: ABOUT
        with gr.Tab("ℹ️ About"):
            gr.Markdown("""
            ## 🏗️ Architecture
            | Component | Technology |
            |-----------|------------|
            | LLM | Google Gemini 1.5 Flash |
            | Agent Framework | LangChain ReAct |
            | Database | SQLite |
            | UI | Gradio |
            | Platform | Google Colab |

            ## 🛠️ Tools & APIs
            | Tool | API |
            |------|-----|
            | Weather | Weatherstack API |
            | Currency | ExchangeRate-API |
            | Flights/Hotels/Search | SERP API |
            | Itinerary Generation | Gemini LLM |
            | Budget Estimation | Built-in |

            ## 📅 Track A | 8-Week Timeline
            - **Week 1-2**: Foundation, RAG chatbot, Gradio UI
            - **Week 3-4**: LangChain Agent + 5 API tools
            - **Week 5-6**: Itinerary, Budget, SQLite DB
            - **Week 7-8**: Testing, polish, final demo

            **Project by:** [Your Name] | **Internship:** 2024
            """)

final_app.launch(share=True, debug=False)
print("\n🎉 FINAL AI Travel Concierge is LIVE!")
print("📌 Share the link above to demo your project!")

/tmp/ipykernel_4655/1754334328.py:53: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AI Travel Concierge", theme=gr.themes.Soft(primary_hue="blue"), css=css) as final_app:
/tmp/ipykernel_4655/1754334328.py:53: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="AI Travel Concierge", theme=gr.themes.Soft(primary_hue="blue"), css=css) as final_app:
/tmp/ipykernel_4655/1754334328.py:66: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  bot = gr.Chatbot(
/tmp/ipykernel_4655/175433432

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b909319a442410ed3e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🎉 FINAL AI Travel Concierge is LIVE!
📌 Share the link above to demo your project!
